In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml kailash-align

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# M6 Ollama bootstrap — local LLMs, no API keys
# On Colab: installs + starts Ollama and pulls this lesson's models.
# Locally: verifies the daemon is running and models are pulled.
# ══════════════════════════════════════════════════════════════════
_LESSON_MODELS = ['qwen2.5:0.5b', 'llama3.2:3b']
import sys, os, time, shutil, subprocess
import httpx

_OLLAMA_BASE_URL = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')
_IN_COLAB = 'google.colab' in sys.modules

def _daemon_up(timeout_s=1.0):
    try:
        r = httpx.get(f'{_OLLAMA_BASE_URL}/api/tags', timeout=timeout_s)
        r.raise_for_status()
        return r.json()
    except Exception:
        return None

if _IN_COLAB and shutil.which('ollama') is None:
    print('[ollama] installing binary …')
    subprocess.run(
        'curl -fsSL https://ollama.com/install.sh | sh',
        shell=True, check=True,
    )

if _IN_COLAB and _daemon_up() is None:
    print('[ollama] starting daemon …')
    subprocess.Popen(
        'nohup ollama serve > /tmp/ollama.log 2>&1 &',
        shell=True,
    )
    deadline = time.monotonic() + 30
    while time.monotonic() < deadline and _daemon_up() is None:
        time.sleep(0.5)

_tags = _daemon_up(timeout_s=2.0)
if _tags is None:
    raise RuntimeError(
        f'Ollama daemon not reachable at {_OLLAMA_BASE_URL}. '
        'On Colab: re-run this cell. Locally: run `ollama serve`.'
    )

_have = {m.get('name','').split(':',1)[0] for m in _tags.get('models', [])}
for _model in _LESSON_MODELS:
    _fam = _model.split(':', 1)[0]
    if _fam in _have:
        continue
    if _IN_COLAB:
        print(f'[ollama] pulling {_model} (one-time) …')
        subprocess.run(['ollama', 'pull', _model], check=True)
    else:
        raise RuntimeError(
            f'Model {_model!r} not pulled. Run: ollama pull {_model}'
        )

print(
    '✓ Ollama ready at', _OLLAMA_BASE_URL,
    '— models:', _LESSON_MODELS,
)



In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp06
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp06/diagnostics/py ──
"""Bias-mitigated LLM-as-judge primitives built on Kaizen ``Delegate``.

All lens classes that need an LLM judge (Output, Retrieval, Alignment) share
this module. The single ``JudgeCallable`` abstraction wraps a Kaizen Delegate
with:

    * position-swap bias mitigation for pairwise preference judging
    * length-normalised pairwise scoring
    * a hard budget (``max_judge_calls``) so diagnostic runs stay cheap
    * structured INFO logs with correlation IDs per :mod:`rules.observability`

Per ``rules/framework-first.md`` MANDATORY: raw ``openai.chat.completions.create``
is BLOCKED — every LLM call goes through ``Delegate.run_sync`` which honours
the configured cost envelope.

Per ``rules/env-models.md``: ``judge_model`` defaults to
``OPENAI_JUDGE_MODEL`` → ``DEFAULT_LLM_MODEL`` → ``OPENAI_PROD_MODEL``. No
hardcoded model names.
"""

import logging
import os
import re
import threading
import time
import uuid
from dataclasses import dataclass
from typing import Any

from dotenv import load_dotenv

logger = logging.getLogger(__name__)


# ════════════════════════════════════════════════════════════════════════
# Model resolution — rules/env-models.md
# ════════════════════════════════════════════════════════════════════════


def resolve_judge_model(explicit: str | None = None) -> str:
    """Return the judge model name per ``rules/env-models.md``.

    Resolution order: explicit arg → ``OLLAMA_JUDGE_MODEL`` →
    ``OLLAMA_CHAT_MODEL`` → bootstrap default. The previous OpenAI fall-throughs
    (``OPENAI_JUDGE_MODEL`` / ``OPENAI_PROD_MODEL``) are honoured if set so
    legacy ``.env`` files still resolve, but the bootstrap default is the
    primary path.
    """
    load_dotenv()
    if explicit:
        return explicit
    for key in (
        "OLLAMA_JUDGE_MODEL",
        "OLLAMA_CHAT_MODEL",
        "OPENAI_JUDGE_MODEL",
        "DEFAULT_LLM_MODEL",
        "OPENAI_PROD_MODEL",
    ):
        val = os.environ.get(key)
        if val:
            return val
    # Final fallback: the M6 bootstrap chat default.

    return DEFAULT_CHAT_MODEL


# ════════════════════════════════════════════════════════════════════════
# Verdict schema
# ════════════════════════════════════════════════════════════════════════


@dataclass(frozen=True)
class JudgeVerdict:
    """Structured judgement output.

    Attributes:
        score: Numeric verdict in ``[0.0, 1.0]``. Higher is better.
        rationale: Plain-language explanation from the judge.
        criteria: The criteria string the judge was asked to score against.
        judge_model: The model that produced the verdict.
        mode: ``"real"`` when a live LLM was called; ``"fake"`` when the
            budget was exhausted or no delegate was available. Per
            ``rules/observability.md`` §3, every data call tags ``mode``.
        latency_ms: Wall-clock duration of the judge call.
    """

    score: float
    rationale: str
    criteria: str
    judge_model: str
    mode: str
    latency_ms: float

    def to_dict(self) -> dict[str, Any]:
        return {
            "score": self.score,
            "rationale": self.rationale,
            "criteria": self.criteria,
            "judge_model": self.judge_model,
            "mode": self.mode,
            "latency_ms": self.latency_ms,
        }


# ════════════════════════════════════════════════════════════════════════
# Judge runner
# ════════════════════════════════════════════════════════════════════════


_SCORE_REGEX = re.compile(r"score\s*[:=]\s*([0-9]*\.?[0-9]+)", re.IGNORECASE)


def _parse_score(raw: str) -> float:
    """Extract a float score in ``[0, 1]`` from a judge's reply.

    The judge prompt asks for ``SCORE: <float in 0..1>``. A well-behaved
    judge returns e.g. ``SCORE: 0.82``. Fallback: look for the first
    0..1-looking float in the text. If no number is found, return ``0.0``
    and emit a WARN per ``rules/observability.md`` §1 (WARN on fallback).
    """
    match = _SCORE_REGEX.search(raw)
    if match:
        try:
            val = float(match.group(1))
            if val > 1.0 and val <= 10.0:
                # Judges sometimes produce 0..10; normalise.
                val = val / 10.0
            return max(0.0, min(1.0, val))
        except ValueError:
            pass
    # Second attempt — any float 0..1 in the text.
    for token in re.findall(r"[0-9]*\.?[0-9]+", raw):
        try:
            f = float(token)
            if 0.0 <= f <= 1.0:
                return f
        except ValueError:
            continue
    logger.warning(
        "judge.parse_fallback",
        extra={"raw_preview": raw[:120], "parsed_score": 0.0},
    )
    return 0.0


class JudgeCallable:
    """Bias-mitigated LLM-as-judge backed by a Kaizen ``Delegate``.

    The judge is constructed lazily: no network call happens at
    ``__init__`` time. The first ``__call__`` instantiates the Delegate.

    Args:
        judge_model: Model name. Resolved via :func:`resolve_judge_model`
            if ``None``.
        max_judge_calls: Hard cap on live judge calls per instance.
            Defaults to ``50`` (≈ <$2 for typical eval). When exhausted,
            subsequent calls return ``mode="fake"`` verdicts and emit WARN.
        delegate: Optional pre-built Delegate. If ``None``, a minimal one
            is constructed lazily on first call.
        sensitive: When ``True``, the prompt / response pair is NOT logged —
            only hashes. Per ``rules/observability.md`` §4 (no secrets).

    Example::

        judge = JudgeCallable(max_judge_calls=10)
        verdict = judge.score(
            "Paris is the capital of France.",
            criteria="factual_accuracy",
            context="User asked: what is the capital of France?",
        )
        print(verdict.score, verdict.rationale)
    """

    _JUDGE_SYSTEM = (
        "You are a rigorous LLM-as-judge. Score the given response against "
        "the stated criteria. Be strict, cite evidence, and output a single "
        "line of the form 'SCORE: <float in 0..1>' followed by a one-sentence "
        "rationale. Do NOT hedge with 0.5 — commit to a verdict."
    )

    def __init__(
        self,
        *,
        judge_model: str | None = None,
        max_judge_calls: int = 50,
        delegate: Any = None,
        sensitive: bool = False,
    ) -> None:
        if max_judge_calls < 0:
            raise ValueError("max_judge_calls must be >= 0")
        self._model_name = resolve_judge_model(judge_model)
        self._max_calls = max_judge_calls
        self._call_count = 0
        self._sensitive = sensitive
        self._delegate = delegate
        self._lock = threading.Lock()

    @property
    def model(self) -> str:
        return self._model_name

    @property
    def calls_consumed(self) -> int:
        return self._call_count

    @property
    def calls_remaining(self) -> int:
        return max(0, self._max_calls - self._call_count)

    def _ensure_delegate(self) -> Any:
        if self._delegate is not None:
            return self._delegate
        # Lazy import — bootstrap is not needed at module import time.

        self._delegate = make_delegate(
            model=self._model_name,
            system_prompt=self._JUDGE_SYSTEM,
            temperature=0.0,
        )
        logger.info(
            "judge.delegate_constructed",
            extra={"judge_model": self._model_name, "provider": "ollama"},
        )
        return self._delegate

    def close(self) -> None:
        """Release the underlying Delegate's network resources."""
        if self._delegate is not None and hasattr(self._delegate, "close"):
            try:
                self._delegate.close()
            except Exception:
                # Cleanup path — zero-tolerance Rule 3 carve-out.
                pass
        self._delegate = None

    # ── Core scoring API ────────────────────────────────────────────────

    def score(
        self,
        response: str,
        *,
        criteria: str,
        context: str = "",
        run_id: str | None = None,
    ) -> JudgeVerdict:
        """Score a single response against a criteria string.

        Args:
            response: The model output under evaluation.
            criteria: Short phrase describing what to score for
                (e.g. ``"factual_accuracy"``, ``"coherence,helpfulness"``).
            context: Optional context — source document, user prompt, etc.
            run_id: Correlation ID per ``rules/observability.md`` §2.
                Auto-generated if ``None``.

        Returns:
            A :class:`JudgeVerdict`.
        """
        if run_id is None:
            run_id = f"judge-{uuid.uuid4().hex[:12]}"
        t0 = time.monotonic()

        with self._lock:
            over_budget = self._call_count >= self._max_calls
            if not over_budget:
                self._call_count += 1

        if over_budget:
            logger.warning(
                "judge.budget_exhausted",
                extra={
                    "run_id": run_id,
                    "max_calls": self._max_calls,
                    "calls": self._call_count,
                },
            )
            return JudgeVerdict(
                score=0.0,
                rationale=(
                    f"BUDGET EXHAUSTED after {self._max_calls} calls. "
                    "Increase max_judge_calls to score further."
                ),
                criteria=criteria,
                judge_model=self._model_name,
                mode="fake",
                latency_ms=(time.monotonic() - t0) * 1000.0,
            )

        prompt = _build_judge_prompt(response, criteria=criteria, context=context)
        log_extra = {
            "run_id": run_id,
            "judge_model": self._model_name,
            "criteria": criteria,
            "source": "kaizen_delegate",
            "mode": "real",
        }
        if not self._sensitive:
            log_extra["response_preview"] = response[:80]
        logger.info("judge.score.start", extra=log_extra)

        try:
            delegate = self._ensure_delegate()
            raw = delegate.run_sync(prompt)
        except Exception as exc:
            logger.exception(
                "judge.score.error",
                extra={"run_id": run_id, "error": str(exc)},
            )
            raise

        score_val = _parse_score(raw)
        # Rationale = everything after the SCORE line (or whole reply if none).
        rationale = _extract_rationale(raw)
        latency = (time.monotonic() - t0) * 1000.0

        logger.info(
            "judge.score.ok",
            extra={
                "run_id": run_id,
                "judge_model": self._model_name,
                "score": score_val,
                "latency_ms": latency,
                "mode": "real",
            },
        )
        return JudgeVerdict(
            score=score_val,
            rationale=rationale,
            criteria=criteria,
            judge_model=self._model_name,
            mode="real",
            latency_ms=latency,
        )

    # ── Pairwise (win-rate) ────────────────────────────────────────────

    def pairwise(
        self,
        a: str,
        b: str,
        *,
        prompt: str,
        criteria: str = "helpfulness,harmlessness,correctness",
        swap_positions: bool = True,
        length_normalise: bool = True,
        run_id: str | None = None,
    ) -> dict[str, Any]:
        """Pairwise preference with position-swap bias mitigation.

        Runs the judge twice (A-then-B and B-then-A) and averages the
        preference. When ``length_normalise`` is ``True``, the prompt
        instructs the judge to discount verbosity.

        Returns:
            A dict with keys ``winner`` (``"a"`` / ``"b"`` / ``"tie"``),
            ``score_a``, ``score_b``, ``margin``, ``mode``, ``latency_ms``.
        """
        if run_id is None:
            run_id = f"pairwise-{uuid.uuid4().hex[:12]}"
        t0 = time.monotonic()

        crit = criteria
        if length_normalise:
            crit = f"{criteria},length_normalised"

        # Forward pass — A vs B.
        forward = self.score(
            response=_pairwise_blob(a=a, b=b, prompt=prompt),
            criteria=crit,
            context="Which response better satisfies the prompt? SCORE >0.5 if A wins, <0.5 if B wins.",
            run_id=f"{run_id}-ab",
        )
        pref_a_forward = forward.score

        if swap_positions:
            reverse = self.score(
                response=_pairwise_blob(a=b, b=a, prompt=prompt),
                criteria=crit,
                context="Which response better satisfies the prompt? SCORE >0.5 if A wins, <0.5 if B wins.",
                run_id=f"{run_id}-ba",
            )
            # reverse.score > 0.5 means "b was preferred" in reversed framing,
            # i.e. original A was preferred → same direction.
            pref_a = 0.5 * pref_a_forward + 0.5 * (1.0 - reverse.score)
            mode = (
                "real"
                if (forward.mode == "real" and reverse.mode == "real")
                else "fake"
            )
        else:
            pref_a = pref_a_forward
            mode = forward.mode

        margin = pref_a - 0.5
        winner = "a" if pref_a > 0.55 else ("b" if pref_a < 0.45 else "tie")
        latency = (time.monotonic() - t0) * 1000.0
        return {
            "winner": winner,
            "pref_a": pref_a,
            "score_a": pref_a,
            "score_b": 1.0 - pref_a,
            "margin": margin,
            "mode": mode,
            "latency_ms": latency,
            "judge_model": self._model_name,
        }


def _build_judge_prompt(response: str, *, criteria: str, context: str = "") -> str:
    """Assemble the judge prompt. Pure string formatting — no LLM reasoning here."""
    return (
        f"[CONTEXT]\n{context or '(none)'}\n\n"
        f"[RESPONSE]\n{response}\n\n"
        f"[CRITERIA]\n{criteria}\n\n"
        "Output format (exact):\n"
        "SCORE: <float in 0..1>\n"
        "REASON: <one sentence>"
    )


def _pairwise_blob(*, a: str, b: str, prompt: str) -> str:
    return f"[PROMPT]\n{prompt}\n\n" f"[RESPONSE A]\n{a}\n\n" f"[RESPONSE B]\n{b}\n"


def _extract_rationale(raw: str) -> str:
    match = re.search(r"reason\s*[:=]\s*(.+)", raw, re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1).strip().splitlines()[0][:400]
    # Fall back to last 200 chars.
    return raw.strip()[-400:]


# ── shared/mlfp06/diagnostics/py ──
"""Shared Plotly theme and small chart helpers.

Matches the M5 Doctor's Bag visual style (clean ``plotly_white`` template,
``steelblue`` / ``firebrick`` / ``orange`` palette) so M5 and M6 dashboards
sit side-by-side in the capstone deliverable.
"""

from typing import Sequence

import plotly.graph_objects as go

__all__ = [
    "TEMPLATE",
    "PRIMARY",
    "WARN",
    "ACCENT",
    "MUTED",
    "PALETTE",
    "empty_figure",
    "style",
    "color_for",
    "bar_with_threshold",
]

TEMPLATE = "plotly_white"
PRIMARY = "steelblue"
WARN = "firebrick"
ACCENT = "orange"
MUTED = "lightgray"

PALETTE: tuple[str, ...] = (
    "steelblue",
    "firebrick",
    "seagreen",
    "darkorange",
    "mediumpurple",
    "teal",
    "goldenrod",
    "slategray",
)


def empty_figure(title: str, note: str = "no data") -> go.Figure:
    """Return a blank Plotly figure with a centred annotation.

    Used when a lens is asked to plot but has no data — never ``None``
    and never raises; a visible "no data" cue beats a blank screen.
    """
    fig = go.Figure()
    fig.update_layout(
        title=f"{title} — {note}",
        template=TEMPLATE,
        annotations=[
            dict(
                text=note,
                xref="paper",
                yref="paper",
                x=0.5,
                y=0.5,
                showarrow=False,
                font=dict(size=14, color=MUTED),
            )
        ],
    )
    return fig


def style(fig: go.Figure, title: str, *, x: str = "", y: str = "") -> go.Figure:
    """Apply the shared MLFP observatory style to a figure in-place."""
    fig.update_layout(
        title=title,
        template=TEMPLATE,
        xaxis_title=x,
        yaxis_title=y,
        hovermode="x unified",
    )
    return fig


def color_for(index: int) -> str:
    return PALETTE[index % len(PALETTE)]


def bar_with_threshold(
    categories: Sequence[str],
    values: Sequence[float],
    *,
    threshold: float | None = None,
    title: str,
    y_label: str = "",
    above_color: str = WARN,
    below_color: str = PRIMARY,
) -> go.Figure:
    """Bar chart where bars above the threshold are flagged in ``above_color``."""
    if threshold is None:
        colors = [below_color] * len(values)
    else:
        colors = [above_color if v > threshold else below_color for v in values]
    fig = go.Figure(go.Bar(x=list(categories), y=list(values), marker_color=colors))
    if threshold is not None:
        fig.add_hline(
            y=threshold,
            line=dict(color=ACCENT, dash="dash"),
            annotation_text=f"threshold={threshold:g}",
        )
    style(fig, title, x="", y=y_label)
    return fig


# ── shared/mlfp06/diagnostics/py ──
"""Trace format shared by Agent and RAG lenses.

Schema follows the OpenTelemetry GenAI semantic conventions (2026-01 draft)
so Langfuse / Langsmith / Phoenix can ingest MLFP traces without translation.

A trace is a JSONL file — one event per line — with a stable schema:

    {"ts": "2026-04-15T12:01:03.142Z", "run_id": "r_abc123",
     "kind": "thought"|"action"|"observation"|"tool_start"|"tool_end"|"token"|"complete"|"error",
     ...kind-specific fields...,
     "cost_usd": 0.0, "latency_ms": 0.0}

This module owns:

    * ``TraceEvent`` dataclass  — one event, serialisable to JSON
    * ``AgentTrace`` class      — in-memory list of events + JSONL writer
    * ``kaizen_events_to_trace``— Kaizen ``StreamEvent`` → ``TraceEvent`` converter
"""

import json
import logging
import uuid
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Iterator

logger = logging.getLogger(__name__)


# ════════════════════════════════════════════════════════════════════════
# Event schema
# ════════════════════════════════════════════════════════════════════════


@dataclass
class TraceEvent:
    """One event in an agent / RAG trace.

    Fields map 1:1 to the JSONL wire format. Optional fields default to
    ``None`` and are dropped from the serialised form.

    Attributes:
        ts: ISO-8601 UTC timestamp.
        run_id: Correlation ID for the enclosing run.
        kind: Event category. One of: ``token``, ``thought``, ``action``,
            ``observation``, ``tool_start``, ``tool_end``, ``complete``,
            ``error``.
        content: Free-text content (thought text, response chunk, etc).
        tool: Name of tool when ``kind`` is ``tool_*``.
        args: Tool arguments as a JSON-serialisable dict.
        result: Tool result preview (truncated for large outputs).
        error: Error string when ``kind == "error"``.
        cost_usd: Cost attributable to this event.
        latency_ms: Latency of the event (tool call, LLM turn, etc).
        tokens_in, tokens_out: Token counts for LLM calls.
        meta: Arbitrary structured metadata.
    """

    ts: str
    run_id: str
    kind: str
    content: str | None = None
    tool: str | None = None
    args: dict[str, Any] | None = None
    result: str | None = None
    error: str | None = None
    cost_usd: float = 0.0
    latency_ms: float = 0.0
    tokens_in: int | None = None
    tokens_out: int | None = None
    meta: dict[str, Any] = field(default_factory=dict)

    def to_jsonl(self) -> str:
        """Serialise to one line of JSON (no trailing newline)."""
        payload = {k: v for k, v in asdict(self).items() if v is not None and v != {}}
        return json.dumps(payload, ensure_ascii=False, default=str)

    @classmethod
    def from_dict(cls, data: dict[str, Any]) -> "TraceEvent":
        return cls(
            ts=data.get("ts", _now_iso()),
            run_id=data.get("run_id", "unknown"),
            kind=data.get("kind", "unknown"),
            content=data.get("content"),
            tool=data.get("tool"),
            args=data.get("args"),
            result=data.get("result"),
            error=data.get("error"),
            cost_usd=float(data.get("cost_usd", 0.0)),
            latency_ms=float(data.get("latency_ms", 0.0)),
            tokens_in=data.get("tokens_in"),
            tokens_out=data.get("tokens_out"),
            meta=data.get("meta", {}) or {},
        )


# ════════════════════════════════════════════════════════════════════════
# AgentTrace — in-memory + JSONL persistence
# ════════════════════════════════════════════════════════════════════════


class AgentTrace:
    """An append-only in-memory trace with optional JSONL persistence.

    Context-manager compatible: any writes are flushed on ``__exit__``.

    Example::

        with AgentTrace(run_id="demo", path=Path("runs/demo.jsonl")) as trace:
            trace.append(TraceEvent(ts=_now_iso(), run_id="demo", kind="thought",
                                    content="Planning..."))
            for k, v in enumerate(range(3)):
                trace.append_simple(kind="action", tool="search", args={"q": str(k)})
    """

    def __init__(
        self,
        *,
        run_id: str | None = None,
        path: Path | str | None = None,
    ) -> None:
        self.run_id = run_id or f"r_{uuid.uuid4().hex[:12]}"
        self._events: list[TraceEvent] = []
        self.path: Path | None = Path(path) if path is not None else None
        self._file: Any = None
        if self.path is not None:
            self.path.parent.mkdir(parents=True, exist_ok=True)
        logger.info(
            "trace.init",
            extra={
                "run_id": self.run_id,
                "path": str(self.path) if self.path else None,
            },
        )

    def __enter__(self) -> "AgentTrace":
        if self.path is not None:
            self._file = self.path.open("a", encoding="utf-8")
        return self

    def __exit__(self, exc_type, exc, tb) -> None:
        self.close()

    def close(self) -> None:
        if self._file is not None:
            try:
                self._file.flush()
                self._file.close()
            except Exception:
                # Cleanup path — zero-tolerance Rule 3 carve-out.
                pass
            self._file = None

    def __iter__(self) -> Iterator[TraceEvent]:
        return iter(self._events)

    def __len__(self) -> int:
        return len(self._events)

    @property
    def events(self) -> list[TraceEvent]:
        return list(self._events)

    def append(self, event: TraceEvent) -> None:
        """Append a fully-constructed event."""
        self._events.append(event)
        if self._file is not None:
            self._file.write(event.to_jsonl())
            self._file.write("\n")

    def append_simple(self, *, kind: str, **kwargs: Any) -> TraceEvent:
        """Construct + append in one step. Returns the event for chaining."""
        event = TraceEvent(ts=_now_iso(), run_id=self.run_id, kind=kind, **kwargs)
        self.append(event)
        return event

    def total_cost_usd(self) -> float:
        return sum(e.cost_usd for e in self._events)

    def total_latency_ms(self) -> float:
        return sum(e.latency_ms for e in self._events)

    def filter_kind(self, kind: str) -> list[TraceEvent]:
        return [e for e in self._events if e.kind == kind]

    # ── Persistence ─────────────────────────────────────────────────────

    def write(self, path: Path | str) -> Path:
        """Write the full trace to a JSONL file. Returns the resolved path."""
        p = Path(path)
        p.parent.mkdir(parents=True, exist_ok=True)
        with p.open("w", encoding="utf-8") as f:
            for event in self._events:
                f.write(event.to_jsonl())
                f.write("\n")
        logger.info(
            "trace.written",
            extra={"run_id": self.run_id, "path": str(p), "events": len(self._events)},
        )
        return p

    @classmethod
    def read(cls, path: Path | str) -> "AgentTrace":
        """Load a JSONL trace from disk."""
        p = Path(path)
        trace = cls(run_id=p.stem)
        with p.open("r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    trace._events.append(TraceEvent.from_dict(json.loads(line)))
                except json.JSONDecodeError as exc:
                    logger.warning(
                        "trace.read.skip_malformed_line",
                        extra={"path": str(p), "line": line_no, "error": str(exc)},
                    )
        logger.info(
            "trace.read",
            extra={"path": str(p), "events": len(trace._events)},
        )
        return trace


# ════════════════════════════════════════════════════════════════════════
# Kaizen StreamEvent → TraceEvent adapter
# ════════════════════════════════════════════════════════════════════════


def kaizen_events_to_trace(
    stream_events: Iterable[Any],
    *,
    run_id: str | None = None,
    path: Path | str | None = None,
) -> AgentTrace:
    """Convert a Kaizen ``StreamEvent`` iterable into an :class:`AgentTrace`.

    Recognised event types (from ``kaizen_agents.events``):
        * ``TextDelta``     → ``kind="token"``
        * ``ToolCallStart`` → ``kind="tool_start"``
        * ``ToolCallEnd``   → ``kind="tool_end"`` or ``kind="error"``
        * ``TurnComplete``  → ``kind="complete"``
        * ``ErrorEvent``    → ``kind="error"``

    The adapter is defensive — any unknown event type is recorded as
    ``kind="meta"`` with the raw class name in ``meta``.
    """
    trace = AgentTrace(run_id=run_id, path=path)
    pending_tool_starts: dict[str, float] = {}

    for event in stream_events:
        cls_name = type(event).__name__
        ts = _now_iso()

        if cls_name == "TextDelta":
            trace.append(
                TraceEvent(
                    ts=ts,
                    run_id=trace.run_id,
                    kind="token",
                    content=getattr(event, "text", ""),
                )
            )
        elif cls_name == "ToolCallStart":
            call_id = getattr(event, "call_id", "unknown")
            pending_tool_starts[call_id] = _monotonic_ms()
            trace.append(
                TraceEvent(
                    ts=ts,
                    run_id=trace.run_id,
                    kind="tool_start",
                    tool=getattr(event, "name", None),
                    meta={"call_id": call_id},
                )
            )
        elif cls_name == "ToolCallEnd":
            call_id = getattr(event, "call_id", "unknown")
            start_ms = pending_tool_starts.pop(call_id, None)
            latency = (_monotonic_ms() - start_ms) if start_ms is not None else 0.0
            err = getattr(event, "error", None)
            trace.append(
                TraceEvent(
                    ts=ts,
                    run_id=trace.run_id,
                    kind="error" if err else "tool_end",
                    tool=getattr(event, "name", None),
                    result=_truncate(getattr(event, "result", None)),
                    error=str(err) if err else None,
                    latency_ms=latency,
                    meta={"call_id": call_id},
                )
            )
        elif cls_name == "TurnComplete":
            usage = getattr(event, "usage", {}) or {}
            trace.append(
                TraceEvent(
                    ts=ts,
                    run_id=trace.run_id,
                    kind="complete",
                    content=_truncate(getattr(event, "text", None)),
                    tokens_in=usage.get("input_tokens") or usage.get("prompt_tokens"),
                    tokens_out=usage.get("output_tokens")
                    or usage.get("completion_tokens"),
                    cost_usd=float(usage.get("cost_usd", 0.0)),
                    meta={"iterations": getattr(event, "iterations", None)},
                )
            )
        elif cls_name == "ErrorEvent":
            trace.append(
                TraceEvent(
                    ts=ts,
                    run_id=trace.run_id,
                    kind="error",
                    error=str(getattr(event, "error", "unknown")),
                    meta={"details": getattr(event, "details", {})},
                )
            )
        else:
            trace.append(
                TraceEvent(
                    ts=ts,
                    run_id=trace.run_id,
                    kind="meta",
                    meta={"stream_event_type": cls_name},
                )
            )
    return trace


# ════════════════════════════════════════════════════════════════════════
# Helpers
# ════════════════════════════════════════════════════════════════════════


def _now_iso() -> str:
    return (
        datetime.now(timezone.utc)
        .isoformat(timespec="milliseconds")
        .replace("+00:00", "Z")
    )


def _monotonic_ms() -> float:
    import time

    return time.monotonic() * 1000.0


def _truncate(value: Any, limit: int = 500) -> str | None:
    if value is None:
        return None
    s = value if isinstance(value, str) else json.dumps(value, default=str)
    if len(s) <= limit:
        return s
    return s[:limit] + f"... <+{len(s) - limit} chars>"


# ── shared/mlfp06/diagnostics/retrieval.py ──
"""Lens 3 — Retrieval Diagnostics (the Endoscope).

Question answered: *Did we fetch the right context, and did the generator use it?*

Wraps ``ragas`` (context-precision, context-recall, faithfulness, answer-
relevancy) and ``trulens-eval`` (groundedness, relevance) as the primary
backends. When either backend is missing the lens degrades **loudly** —
the method raises a descriptive ``ImportError`` naming the extra to
install, per ``rules/dependencies.md`` ("Optional Extras with Loud
Failure"). Silent ``None`` fallbacks are BLOCKED.

All LLM-as-judge calls for answer-quality scoring route through the
shared :class:`~shared.mlfp06.diagnostics.JudgeCallable` — no raw
``openai.*`` per ``rules/framework-first.md``.

Quick start::


    rag = RAGDiagnostics(max_judge_calls=20)
    df = rag.evaluate(
        queries=["What is photosynthesis?"],
        retrieved_contexts=[[doc1, doc2, doc3]],
        answers=["Photosynthesis is ..."],
        ground_truth_ids=[["doc_42"]],
    )
    board = rag.compare_retrievers(
        retrievers={"bm25": bm25_fn, "dense": dense_fn, "hybrid": hybrid_fn},
        eval_set=eval_set,
        k=5,
    )
    rag.plot_rag_dashboard().show()
    print(rag.report())
"""

import logging
import uuid
from dataclasses import dataclass
from typing import Any, Callable, Sequence

import plotly.graph_objects as go
import polars as pl


logger = logging.getLogger(__name__)

__all__ = ["RAGDiagnostics"]


# Retriever callable: (query, k) -> list of (doc_id, content, score).
RetrievedDoc = tuple[str, str, float]
Retriever = Callable[[str, int], Sequence[RetrievedDoc]]


@dataclass
class _EvalEntry:
    query: str
    answer: str
    retrieved_ids: list[str]
    ground_truth_ids: list[str]
    context_utilisation: float
    faithfulness: float
    recall_at_k: float
    precision_at_k: float
    mode: str


class RAGDiagnostics:
    """Retrieval-lens diagnostics — recall@k, context utilisation, retriever leaderboard.

    Args:
        judge_model: Judge model. Resolved via
            :func:`~shared.mlfp06.diagnostics.resolve_judge_model`.
        max_judge_calls: Hard cap on live judge calls (default ``50``).
        delegate: Optional pre-built Kaizen ``Delegate`` reused by the judge.
        sensitive: When ``True``, query/answer bodies are not logged.
    """

    def __init__(
        self,
        *,
        judge_model: str | None = None,
        max_judge_calls: int = 50,
        delegate: Any = None,
        sensitive: bool = False,
    ) -> None:
        self._judge = JudgeCallable(
            judge_model=judge_model,
            max_judge_calls=max_judge_calls,
            delegate=delegate,
            sensitive=sensitive,
        )
        self._eval_log: list[_EvalEntry] = []
        self._retriever_log: list[dict[str, Any]] = []
        self._sensitive = sensitive
        logger.info(
            "rag_diagnostics.init",
            extra={"judge_model": self._judge.model, "max_calls": max_judge_calls},
        )

    def __enter__(self) -> "RAGDiagnostics":
        return self

    def __exit__(self, *exc: Any) -> None:
        self.close()

    def close(self) -> None:
        self._judge.close()

    # ── Core evaluation API ────────────────────────────────────────────

    def evaluate(
        self,
        queries: Sequence[str],
        retrieved_contexts: Sequence[Sequence[str]],
        answers: Sequence[str],
        *,
        ground_truth_ids: Sequence[Sequence[str]] | None = None,
        retrieved_ids: Sequence[Sequence[str]] | None = None,
        k: int = 5,
        run_id: str | None = None,
    ) -> pl.DataFrame:
        """Score a batch of RAG outputs end-to-end.

        Computes per-query recall@k, precision@k, context-utilisation, and
        faithfulness. When ``ragas`` is installed, its implementations are
        used; otherwise the lens falls back to judge-only scoring and logs
        the fallback at WARN level.

        Args:
            queries: User queries.
            retrieved_contexts: For each query, the ordered list of
                retrieved chunk contents.
            answers: The generator's final answers.
            ground_truth_ids: Optional list of per-query relevant doc IDs
                (required for recall@k / precision@k).
            retrieved_ids: Optional list of per-query retrieved doc IDs in
                the same order as ``retrieved_contexts``. When ``None``
                the lens treats the context strings themselves as IDs.
            k: Cut-off used for recall@k / precision@k.
            run_id: Correlation ID per ``rules/observability.md``.

        Returns:
            A Polars DataFrame with one row per query.
        """
        n = len(queries)
        if not (len(retrieved_contexts) == n == len(answers)):
            raise ValueError(
                "queries, retrieved_contexts, answers must all be same length"
            )
        run_id = run_id or f"rag_eval-{uuid.uuid4().hex[:12]}"
        logger.info(
            "rag.evaluate.start",
            extra={"run_id": run_id, "n": n, "k": k, "mode": "real"},
        )

        ragas_scores = _try_ragas_evaluate(
            queries=queries,
            retrieved_contexts=retrieved_contexts,
            answers=answers,
            ground_truth_ids=ground_truth_ids,
        )
        rows: list[dict[str, Any]] = []
        for i in range(n):
            ids_i = (
                list(retrieved_ids[i])
                if retrieved_ids is not None
                else list(retrieved_contexts[i])
            )
            truth_i = list(ground_truth_ids[i]) if ground_truth_ids is not None else []
            recall = _recall_at_k(retrieved=ids_i[:k], relevant=truth_i)
            precision = _precision_at_k(retrieved=ids_i[:k], relevant=truth_i)

            if ragas_scores is not None:
                faithfulness = float(ragas_scores["faithfulness"][i])
                context_util = float(ragas_scores["context_precision"][i])
                backend_mode = "real"
            else:
                faithfulness_verdict = self._judge.score(
                    response=answers[i],
                    criteria="faithfulness,grounded_in_context,no_fabrication",
                    context=(
                        "[QUERY]\n"
                        + queries[i]
                        + "\n\n[RETRIEVED CONTEXT]\n"
                        + "\n\n".join(retrieved_contexts[i])
                    ),
                    run_id=f"{run_id}-faith-{i}",
                )
                faithfulness = faithfulness_verdict.score
                context_util = _heuristic_context_utilisation(
                    answer=answers[i], contexts=retrieved_contexts[i]
                )
                backend_mode = faithfulness_verdict.mode

            self._eval_log.append(
                _EvalEntry(
                    query=queries[i],
                    answer=answers[i],
                    retrieved_ids=ids_i[:k],
                    ground_truth_ids=truth_i,
                    context_utilisation=context_util,
                    faithfulness=faithfulness,
                    recall_at_k=recall,
                    precision_at_k=precision,
                    mode=backend_mode,
                )
            )
            rows.append(
                {
                    "idx": i,
                    "recall_at_k": recall,
                    "precision_at_k": precision,
                    "context_utilisation": context_util,
                    "faithfulness": faithfulness,
                    "k": k,
                    "mode": backend_mode,
                }
            )
        df = pl.DataFrame(rows)
        logger.info(
            "rag.evaluate.ok",
            extra={
                "run_id": run_id,
                "n": n,
                "mean_recall": float(df["recall_at_k"].mean() or 0.0),
                "mean_faithfulness": float(df["faithfulness"].mean() or 0.0),
                "source": "ragas" if ragas_scores is not None else "judge_fallback",
                "mode": "real",
            },
        )
        return df

    # ── Retriever leaderboard ──────────────────────────────────────────

    def compare_retrievers(
        self,
        retrievers: dict[str, Retriever],
        eval_set: Sequence[dict[str, Any]],
        *,
        k: int = 5,
        run_id: str | None = None,
    ) -> pl.DataFrame:
        """Leaderboard over BM25 / dense / hybrid / HyDE / ... retrievers.

        Each element of ``eval_set`` must have keys:

            * ``query`` (str)
            * ``relevant_ids`` (list[str]) — ground-truth doc IDs

        ``retrievers`` maps a short label to a callable
        ``(query, k) -> [(doc_id, content, score), ...]``.

        Returns a DataFrame sorted by ``mrr`` descending.
        """
        run_id = run_id or f"retriever_cmp-{uuid.uuid4().hex[:12]}"
        if not retrievers:
            raise ValueError("retrievers dict must be non-empty")
        if not eval_set:
            raise ValueError("eval_set must be non-empty")
        logger.info(
            "rag.compare_retrievers.start",
            extra={
                "run_id": run_id,
                "retrievers": list(retrievers),
                "n_queries": len(eval_set),
                "k": k,
                "mode": "real",
            },
        )

        rows: list[dict[str, Any]] = []
        for name, fn in retrievers.items():
            per_query: list[dict[str, float]] = []
            for entry in eval_set:
                query = entry["query"]
                relevant = list(entry.get("relevant_ids") or [])
                hits = list(fn(query, k)) or []
                retrieved_ids = [h[0] for h in hits[:k]]
                per_query.append(
                    {
                        "recall_at_k": _recall_at_k(retrieved_ids, relevant),
                        "precision_at_k": _precision_at_k(retrieved_ids, relevant),
                        "mrr": _reciprocal_rank(retrieved_ids, relevant),
                        "ndcg_at_k": _ndcg_at_k(retrieved_ids, relevant, k),
                    }
                )
            # Aggregate over queries.
            agg = {
                "retriever": name,
                "recall_at_k": _mean([r["recall_at_k"] for r in per_query]),
                "precision_at_k": _mean([r["precision_at_k"] for r in per_query]),
                "mrr": _mean([r["mrr"] for r in per_query]),
                "ndcg_at_k": _mean([r["ndcg_at_k"] for r in per_query]),
                "n": len(per_query),
                "k": k,
            }
            rows.append(agg)
            self._retriever_log.append({**agg, "run_id": run_id})
        board = pl.DataFrame(rows).sort("mrr", descending=True)
        logger.info(
            "rag.compare_retrievers.ok",
            extra={
                "run_id": run_id,
                "winner": str(board["retriever"][0]) if board.height else None,
                "mode": "real",
            },
        )
        return board

    # ── Individual metric helpers (public) ─────────────────────────────

    def recall_at_k(
        self,
        retrieved_ids: Sequence[str],
        relevant_ids: Sequence[str],
        *,
        k: int = 5,
    ) -> float:
        """Recall@k — fraction of the relevant set captured in top-k."""
        return _recall_at_k(list(retrieved_ids)[:k], list(relevant_ids))

    def precision_at_k(
        self,
        retrieved_ids: Sequence[str],
        relevant_ids: Sequence[str],
        *,
        k: int = 5,
    ) -> float:
        """Precision@k — fraction of top-k that is relevant."""
        return _precision_at_k(list(retrieved_ids)[:k], list(relevant_ids))

    def context_utilisation(
        self,
        answer: str,
        contexts: Sequence[str],
    ) -> float:
        """How much of the answer's content is traceable to retrieved context.

        Uses a token-overlap heuristic (fast, local, no LLM call). For a
        judge-based evaluation use
        :meth:`~shared.mlfp06.diagnostics.LLMDiagnostics.faithfulness`.
        """
        return _heuristic_context_utilisation(answer=answer, contexts=contexts)

    def ragas_scores(
        self,
        queries: Sequence[str],
        retrieved_contexts: Sequence[Sequence[str]],
        answers: Sequence[str],
        *,
        ground_truth_ids: Sequence[Sequence[str]] | None = None,
    ) -> pl.DataFrame:
        """Run the full RAGAS evaluation (requires ``ragas``).

        Raises ``ImportError`` with an actionable message when ragas is
        not installed — per ``rules/dependencies.md`` "Optional Extras
        with Loud Failure".
        """
        scores = _try_ragas_evaluate(
            queries=queries,
            retrieved_contexts=retrieved_contexts,
            answers=answers,
            ground_truth_ids=ground_truth_ids,
        )
        if scores is None:
            raise ImportError(
                "ragas is required for ragas_scores(). Install with "
                "`pip install ragas>=0.2`."
            )
        return pl.DataFrame(scores)

    # Note: A `trulens_scores()` method previously routed groundedness and
    # answer-relevance through trulens-eval + OpenAI. It was removed in the
    # 2026-04-20 sync because:
    #   1. trulens-dashboard pins psutil<6, blocking kailash>=2.8.9 (psutil>=7).
    #   2. The implementation routed through OpenAI, BLOCKED by Redline 14
    #      (M6 is Ollama-only).
    # Equivalent metrics (faithfulness + answer relevance) remain available via
    # `ragas_scores()` above, which uses the M6-mandated Ollama provider.

    # ── DataFrames ─────────────────────────────────────────────────────

    def eval_df(self) -> pl.DataFrame:
        """One row per :meth:`evaluate` sample."""
        if not self._eval_log:
            return pl.DataFrame(
                schema={
                    "query_preview": pl.Utf8,
                    "recall_at_k": pl.Float64,
                    "precision_at_k": pl.Float64,
                    "context_utilisation": pl.Float64,
                    "faithfulness": pl.Float64,
                    "mode": pl.Utf8,
                }
            )
        return pl.DataFrame(
            [
                {
                    "query_preview": "<redacted>" if self._sensitive else e.query[:120],
                    "recall_at_k": e.recall_at_k,
                    "precision_at_k": e.precision_at_k,
                    "context_utilisation": e.context_utilisation,
                    "faithfulness": e.faithfulness,
                    "mode": e.mode,
                }
                for e in self._eval_log
            ]
        )

    def retriever_df(self) -> pl.DataFrame:
        """Retriever leaderboard history."""
        if not self._retriever_log:
            return pl.DataFrame(
                schema={
                    "retriever": pl.Utf8,
                    "recall_at_k": pl.Float64,
                    "precision_at_k": pl.Float64,
                    "mrr": pl.Float64,
                    "ndcg_at_k": pl.Float64,
                    "n": pl.Int64,
                    "k": pl.Int64,
                }
            )
        return pl.DataFrame(self._retriever_log).drop("run_id", strict=False)

    # ── Plots ──────────────────────────────────────────────────────────

    def plot_rag_dashboard(self) -> go.Figure:
        """2x2 dashboard: recall@k curve, context-util histogram, faithfulness
        scatter, retriever leaderboard."""
        from plotly.subplots import make_subplots

        if not self._eval_log and not self._retriever_log:
            return empty_figure("Retrieval Lens Dashboard (Endoscope)")

        fig = make_subplots(
            rows=2,
            cols=2,
            subplot_titles=(
                "Recall@k per query",
                "Context utilisation histogram",
                "Faithfulness vs context-utilisation",
                "Retriever leaderboard (MRR)",
            ),
        )

        eval_df = self.eval_df()
        if eval_df.height:
            fig.add_trace(
                go.Scatter(
                    x=list(range(eval_df.height)),
                    y=eval_df["recall_at_k"].to_list(),
                    mode="lines+markers",
                    marker=dict(color=PRIMARY),
                    name="recall@k",
                ),
                row=1,
                col=1,
            )
            fig.add_trace(
                go.Histogram(
                    x=eval_df["context_utilisation"].to_list(),
                    marker_color=ACCENT,
                    nbinsx=20,
                ),
                row=1,
                col=2,
            )
            fig.add_trace(
                go.Scatter(
                    x=eval_df["context_utilisation"].to_list(),
                    y=eval_df["faithfulness"].to_list(),
                    mode="markers",
                    marker=dict(color=WARN, size=8),
                ),
                row=2,
                col=1,
            )
        board = self.retriever_df()
        if board.height:
            fig.add_trace(
                go.Bar(
                    x=board["retriever"].to_list(),
                    y=board["mrr"].to_list(),
                    marker_color=PRIMARY,
                ),
                row=2,
                col=2,
            )

        fig.update_layout(
            title="Retrieval Lens Dashboard (Endoscope)",
            template=TEMPLATE,
            showlegend=False,
            height=640,
        )
        return fig

    # ── Report ─────────────────────────────────────────────────────────

    def report(self) -> str:
        """Plain-text Prescription Pad for the retrieval lens."""
        out: list[str] = []
        eval_df = self.eval_df()
        if eval_df.height:
            mean_r = float(eval_df["recall_at_k"].mean() or 0.0)
            mean_p = float(eval_df["precision_at_k"].mean() or 0.0)
            mean_u = float(eval_df["context_utilisation"].mean() or 0.0)
            mean_f = float(eval_df["faithfulness"].mean() or 0.0)
            out.append(
                f"evaluate: {eval_df.height} queries, recall@k={mean_r:.2f}, "
                f"precision@k={mean_p:.2f}, context_util={mean_u:.2f}, "
                f"faithfulness={mean_f:.2f}"
            )
            if mean_r < 0.5:
                out.append(
                    "  -> recall below 0.5: widen top-k, add HyDE, or retune embeddings"
                )
            if mean_u < 0.4:
                out.append(
                    "  -> context utilisation below 0.4: answers ignore retrieved context; "
                    "consider reranking or prompt changes"
                )
            if mean_f < 0.7:
                out.append(
                    "  -> faithfulness below 0.7: model is inventing — add citation constraint"
                )
        board = self.retriever_df()
        if board.height:
            top = board.row(0, named=True)
            out.append(
                f"retrievers: top={top['retriever']} "
                f"(mrr={top['mrr']:.2f}, ndcg@k={top['ndcg_at_k']:.2f})"
            )
        if not out:
            return "retrieval-lens: no readings recorded yet."
        return "retrieval-lens:\n  " + "\n  ".join(out)


# ════════════════════════════════════════════════════════════════════════
# Metric helpers — pure, no LLM calls
# ════════════════════════════════════════════════════════════════════════


def _recall_at_k(retrieved: Sequence[str], relevant: Sequence[str]) -> float:
    if not relevant:
        return 0.0
    rset = set(relevant)
    hits = sum(1 for r in retrieved if r in rset)
    return hits / len(rset)


def _precision_at_k(retrieved: Sequence[str], relevant: Sequence[str]) -> float:
    if not retrieved:
        return 0.0
    rset = set(relevant)
    hits = sum(1 for r in retrieved if r in rset)
    return hits / len(retrieved)


def _reciprocal_rank(retrieved: Sequence[str], relevant: Sequence[str]) -> float:
    rset = set(relevant)
    for idx, doc in enumerate(retrieved, start=1):
        if doc in rset:
            return 1.0 / idx
    return 0.0


def _ndcg_at_k(retrieved: Sequence[str], relevant: Sequence[str], k: int) -> float:
    import math

    rset = set(relevant)
    dcg = 0.0
    for idx, doc in enumerate(retrieved[:k], start=1):
        if doc in rset:
            dcg += 1.0 / math.log2(idx + 1)
    ideal_hits = min(len(rset), k)
    if ideal_hits == 0:
        return 0.0
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg if idcg else 0.0


def _mean(xs: Sequence[float]) -> float:
    return sum(xs) / len(xs) if xs else 0.0


def _heuristic_context_utilisation(answer: str, contexts: Sequence[str]) -> float:
    """Token-overlap context utilisation in ``[0, 1]``.

    Fraction of answer tokens (non-stopword, length >= 4) that appear in
    at least one retrieved context. Not an LLM judgement — see
    :meth:`LLMDiagnostics.faithfulness` for that.
    """
    _STOP = {
        "the",
        "that",
        "this",
        "with",
        "from",
        "have",
        "they",
        "their",
        "them",
        "these",
        "those",
        "into",
        "been",
        "were",
        "will",
        "would",
        "about",
        "which",
        "there",
        "where",
    }
    ans_tokens = {
        t
        for t in answer.lower().split()
        if len(t) >= 4 and t.isalpha() and t not in _STOP
    }
    if not ans_tokens:
        return 0.0
    context_blob = " ".join(contexts).lower()
    grounded = sum(1 for t in ans_tokens if t in context_blob)
    return grounded / len(ans_tokens)


def _try_ragas_evaluate(
    *,
    queries: Sequence[str],
    retrieved_contexts: Sequence[Sequence[str]],
    answers: Sequence[str],
    ground_truth_ids: Sequence[Sequence[str]] | None,
) -> dict[str, list[float]] | None:
    """Call RAGAS if available; return ``None`` when it isn't (caller falls back).

    Per ``rules/dependencies.md`` the fallback is allowed because
    ``RAGDiagnostics.evaluate`` loudly surfaces the fallback via a WARN
    log line, and the public-facing :meth:`RAGDiagnostics.ragas_scores`
    method raises an ``ImportError`` naming the extra.
    """
    try:
        from ragas import evaluate as ragas_evaluate  # type: ignore[import-not-found]
        from ragas.metrics import (  # type: ignore[import-not-found]
            answer_relevancy,
            context_precision,
            context_recall,
            faithfulness as ragas_faithfulness,
        )
        from datasets import Dataset  # type: ignore[import-not-found]
    except ImportError:
        logger.warning(
            "rag.evaluate.ragas_unavailable",
            extra={"reason": "ragas or datasets not installed", "mode": "real"},
        )
        return None

    try:
        ds = Dataset.from_dict(
            {
                "question": list(queries),
                "contexts": [list(c) for c in retrieved_contexts],
                "answer": list(answers),
                "ground_truth": [
                    ", ".join(gt) if gt else ""
                    for gt in (ground_truth_ids or [[] for _ in queries])
                ],
            }
        )
        metrics = [ragas_faithfulness, context_precision, answer_relevancy]
        if ground_truth_ids is not None:
            metrics.append(context_recall)
        result = ragas_evaluate(ds, metrics=metrics)
    except Exception as exc:  # pragma: no cover — ragas internal error
        logger.warning(
            "rag.evaluate.ragas_error",
            extra={"error": str(exc), "mode": "real"},
        )
        return None

    # ``result`` is a RagasResult whose ``.scores`` attribute holds a list of
    # per-row dicts. Fall back to ``.to_pandas()`` on older versions.
    try:
        rows = list(result.scores)  # type: ignore[attr-defined]
    except AttributeError:  # pragma: no cover
        rows = result.to_pandas().to_dict("records")  # type: ignore[attr-defined]

    def _col(key: str) -> list[float]:
        return [float(r.get(key, 0.0)) for r in rows]

    return {
        "faithfulness": _col("faithfulness"),
        "context_precision": _col("context_precision"),
        "context_recall": (
            _col("context_recall") if ground_truth_ids else [0.0] * len(rows)
        ),
        "answer_relevancy": _col("answer_relevancy"),
    }


# ── shared/mlfp06/diagnostics/output.py ──
"""Lens 1 — Output Diagnostics (the Stethoscope).

Question answered: *Is the generation coherent, faithful, and on-task?*

Wraps industry evaluation libraries (``deepeval``, ``ragas``, ``sacrebleu``,
``rouge-score``, ``bert-score``) plus a Kaizen-powered LLM-as-judge. Raw
``openai.*`` calls are BLOCKED per ``rules/framework-first.md``.

Typical use::


    diag = LLMDiagnostics(max_judge_calls=20)
    verdict = diag.llm_as_judge(
        prompt="Capital of France?",
        response="Paris.",
        criteria="factual_accuracy",
    )
    faithful_df = diag.faithfulness(response, context=["Paris is the capital..."])
    diag.plot_output_dashboard().show()
    print(diag.report())
"""

import logging
import math
import uuid
from collections import Counter
from dataclasses import dataclass
from typing import Any, Iterable, Sequence

import plotly.graph_objects as go
import polars as pl


logger = logging.getLogger(__name__)

__all__ = ["LLMDiagnostics", "JudgeVerdict"]

# Refusal detection patterns — used ONLY as a heuristic for `refusal_rate`.
# Per rules/agent-reasoning.md: this is not agent decision-making, it's a
# metric heuristic over already-generated outputs. The LLM judge is used
# for the nuanced calls.
_REFUSAL_MARKERS: tuple[str, ...] = (
    "i can't",
    "i cannot",
    "i'm unable",
    "i am unable",
    "i won't",
    "i will not",
    "as an ai",
    "i'm not able",
    "refuse",
    "not appropriate",
    "unable to help",
)


@dataclass
class _OutputEntry:
    prompt: str
    response: str
    criteria: str
    verdict: JudgeVerdict


class LLMDiagnostics:
    """Output-lens diagnostics — faithfulness, coherence, refusal calibration.

    Args:
        judge_model: Judge model name. Resolved via
            :func:`~shared.mlfp06.diagnostics.resolve_judge_model`.
        max_judge_calls: Hard cap on live judge calls (default ``50``).
        delegate: Optional pre-built Kaizen ``Delegate`` (used as the judge).
        sensitive: When ``True``, prompt/response bodies are not logged.
    """

    def __init__(
        self,
        *,
        judge_model: str | None = None,
        max_judge_calls: int = 50,
        delegate: Any = None,
        sensitive: bool = False,
    ) -> None:
        self._judge = JudgeCallable(
            judge_model=judge_model,
            max_judge_calls=max_judge_calls,
            delegate=delegate,
            sensitive=sensitive,
        )
        self._log: list[_OutputEntry] = []
        self._refusal_log: list[dict[str, Any]] = []
        self._faithful_log: list[dict[str, Any]] = []
        self._consistency_log: list[dict[str, Any]] = []
        logger.info(
            "output_diagnostics.init",
            extra={"judge_model": self._judge.model, "max_calls": max_judge_calls},
        )

    def __enter__(self) -> "LLMDiagnostics":
        return self

    def __exit__(self, *exc: Any) -> None:
        self.close()

    def close(self) -> None:
        self._judge.close()

    # ── Judge (Kaizen-backed) ──────────────────────────────────────────

    def llm_as_judge(
        self,
        prompt: str,
        response: str,
        *,
        criteria: str = "coherence,helpfulness,harmlessness",
        run_id: str | None = None,
    ) -> JudgeVerdict:
        """Score a single (prompt, response) pair via the Kaizen-backed judge.

        Args:
            prompt: The user prompt the model was responding to.
            response: The model's response under evaluation.
            criteria: Comma-separated scoring criteria.
            run_id: Correlation ID per ``rules/observability.md``.

        Returns:
            A :class:`~shared.mlfp06.diagnostics.JudgeVerdict`.
        """
        run_id = run_id or f"llm_judge-{uuid.uuid4().hex[:12]}"
        verdict = self._judge.score(
            response,
            criteria=criteria,
            context=f"User prompt: {prompt}",
            run_id=run_id,
        )
        self._log.append(
            _OutputEntry(
                prompt=prompt, response=response, criteria=criteria, verdict=verdict
            )
        )
        return verdict

    def evaluate(
        self,
        prompts: Sequence[str],
        responses: Sequence[str],
        *,
        criteria: str = "coherence,helpfulness,harmlessness",
        run_id: str | None = None,
    ) -> pl.DataFrame:
        """Batch judge over a sequence of (prompt, response) pairs.

        Returns a Polars DataFrame with one row per pair.
        """
        if len(prompts) != len(responses):
            raise ValueError("prompts and responses must be same length")
        run_id = run_id or f"llm_eval-{uuid.uuid4().hex[:12]}"
        rows: list[dict[str, Any]] = []
        for i, (p, r) in enumerate(zip(prompts, responses)):
            v = self.llm_as_judge(p, r, criteria=criteria, run_id=f"{run_id}-{i}")
            rows.append(
                {
                    "idx": i,
                    "score": v.score,
                    "mode": v.mode,
                    "latency_ms": v.latency_ms,
                    "criteria": v.criteria,
                    "judge_model": v.judge_model,
                    "rationale": v.rationale,
                }
            )
        return pl.DataFrame(rows)

    # ── Faithfulness (RAG grounding) ───────────────────────────────────

    def faithfulness(
        self,
        response: str,
        context: Sequence[str] | str,
        *,
        run_id: str | None = None,
    ) -> pl.DataFrame:
        """Judge whether ``response`` is grounded in ``context``.

        Delegates to the Kaizen judge with a strict grounding criterion.
        Returns a one-row Polars DataFrame. Multi-chunk context is joined
        with explicit chunk markers so the judge can cite.
        """
        if isinstance(context, str):
            context_blob = context
        else:
            context_blob = "\n\n".join(
                f"[chunk {i}] {c}" for i, c in enumerate(context)
            )
        verdict = self._judge.score(
            response,
            criteria="faithfulness,grounded_in_context,no_fabrication",
            context=f"Retrieved context:\n{context_blob}",
            run_id=run_id or f"faithful-{uuid.uuid4().hex[:12]}",
        )
        row = {
            "faithfulness": verdict.score,
            "mode": verdict.mode,
            "judge_model": verdict.judge_model,
            "latency_ms": verdict.latency_ms,
            "rationale": verdict.rationale,
        }
        self._faithful_log.append(row)
        return pl.DataFrame([row])

    # ── Self-consistency / hallucination scan ──────────────────────────

    def self_consistency(
        self,
        responses: Sequence[str],
        *,
        prompt: str = "",
        run_id: str | None = None,
    ) -> pl.DataFrame:
        """Measure agreement across ``n`` samples from the same prompt.

        The caller samples the responses externally (typically via
        ``Delegate.run_sync`` called ``n`` times with a non-zero
        temperature). This method then:

            * normalises whitespace + lowercases tokens
            * computes pairwise ROUGE-L (simple Jaccard fallback when
              ``rouge-score`` is not installed)
            * flags hallucination candidates as the lowest-agreement sample

        Returns a Polars DataFrame with ``idx``, ``response`` (truncated),
        ``agreement`` (mean pairwise similarity), ``is_outlier``.
        """
        if len(responses) < 2:
            raise ValueError("self_consistency needs >= 2 samples")
        run_id = run_id or f"self_consistency-{uuid.uuid4().hex[:12]}"

        similarity_fn = _build_similarity_fn()
        agreements: list[float] = []
        for i, ri in enumerate(responses):
            others = [rj for j, rj in enumerate(responses) if j != i]
            sims = [similarity_fn(ri, rj) for rj in others]
            agreements.append(sum(sims) / max(len(sims), 1))

        mean_agreement = sum(agreements) / len(agreements)
        threshold = mean_agreement * 0.75

        rows = [
            {
                "idx": i,
                "response_preview": r[:120],
                "agreement": agreements[i],
                "is_outlier": agreements[i] < threshold,
            }
            for i, r in enumerate(responses)
        ]
        self._consistency_log.append(
            {
                "run_id": run_id,
                "prompt_preview": prompt[:120],
                "n_samples": len(responses),
                "mean_agreement": mean_agreement,
                "n_outliers": sum(1 for a in agreements if a < threshold),
            }
        )
        logger.info(
            "output.self_consistency",
            extra={
                "run_id": run_id,
                "n_samples": len(responses),
                "mean_agreement": mean_agreement,
                "source": "local_metric",
                "mode": "real",
            },
        )
        return pl.DataFrame(rows)

    # ── Refusal calibration ────────────────────────────────────────────

    def refusal_rate(
        self,
        responses: Iterable[str],
        *,
        label: str = "sample",
    ) -> float:
        """Fraction of responses that look like a refusal.

        This is a heuristic over already-generated outputs (see
        ``_REFUSAL_MARKERS``) — it is *not* agent decision-making, so the
        ``rules/agent-reasoning.md`` keyword-match prohibition does not
        apply. For nuanced calls use :meth:`llm_as_judge` with
        ``criteria="is_refusal"``.
        """
        responses = list(responses)
        if not responses:
            return 0.0
        refused = sum(1 for r in responses if _looks_like_refusal(r))
        rate = refused / len(responses)
        self._refusal_log.append(
            {"label": label, "n": len(responses), "refused": refused, "rate": rate}
        )
        logger.info(
            "output.refusal_rate",
            extra={"label": label, "rate": rate, "n": len(responses), "mode": "real"},
        )
        return rate

    def over_refusal(self, benign_responses: Iterable[str]) -> float:
        """Refusal rate on a benign set — any refusals are over-refusals."""
        return self.refusal_rate(benign_responses, label="benign")

    # ── Classical metrics (thin wrappers) ──────────────────────────────

    def rouge(
        self,
        predictions: Sequence[str],
        references: Sequence[str],
        *,
        rouge_type: str = "rougeL",
    ) -> pl.DataFrame:
        """ROUGE (default ``rougeL``) score per (prediction, reference) pair."""
        if len(predictions) != len(references):
            raise ValueError("predictions and references must be same length")
        try:
            from rouge_score import rouge_scorer
        except ImportError as exc:  # pragma: no cover — optional extra
            raise ImportError(
                "rouge requires the rouge-score package. Install with "
                "`pip install rouge-score`."
            ) from exc
        scorer = rouge_scorer.RougeScorer([rouge_type], use_stemmer=True)
        rows = []
        for i, (p, r) in enumerate(zip(predictions, references)):
            s = scorer.score(r, p)[rouge_type]
            rows.append(
                {
                    "idx": i,
                    "precision": s.precision,
                    "recall": s.recall,
                    "fmeasure": s.fmeasure,
                }
            )
        return pl.DataFrame(rows)

    def bleu(
        self,
        predictions: Sequence[str],
        references: Sequence[str],
    ) -> float:
        """Corpus-level BLEU via ``sacrebleu``."""
        if len(predictions) != len(references):
            raise ValueError("predictions and references must be same length")
        try:
            import sacrebleu
        except ImportError as exc:  # pragma: no cover
            raise ImportError(
                "bleu requires sacrebleu. Install with `pip install sacrebleu`."
            ) from exc
        # sacrebleu expects list-of-references (list of corpora), not pairs.
        return float(sacrebleu.corpus_bleu(list(predictions), [list(references)]).score)

    def bertscore(
        self,
        predictions: Sequence[str],
        references: Sequence[str],
        *,
        lang: str = "en",
    ) -> pl.DataFrame:
        """BERTScore per pair (requires the ``bert-score`` extra)."""
        try:
            from bert_score import score as _bs
        except ImportError as exc:  # pragma: no cover
            raise ImportError(
                "bertscore requires bert-score. Install with "
                "`pip install bert-score`."
            ) from exc
        P, R, F = _bs(list(predictions), list(references), lang=lang, verbose=False)
        return pl.DataFrame(
            {
                "idx": list(range(len(predictions))),
                "precision": [float(x) for x in P.tolist()],
                "recall": [float(x) for x in R.tolist()],
                "f1": [float(x) for x in F.tolist()],
            }
        )

    def perplexity(self, token_logprobs: Sequence[float]) -> float:
        """Perplexity from precomputed token log-probabilities (natural log).

        The caller supplies the logprobs (typically from the target model's
        ``logprobs`` API). This method does not call the model — it is a
        pure reduction so the judge can focus on grading, not scoring.
        """
        if not token_logprobs:
            return float("nan")
        mean = sum(token_logprobs) / len(token_logprobs)
        try:
            return float(math.exp(-mean))
        except OverflowError:
            return float("inf")

    # ── DataFrames ─────────────────────────────────────────────────────

    def results_df(self) -> pl.DataFrame:
        """One row per :meth:`llm_as_judge` call."""
        if not self._log:
            return pl.DataFrame(
                schema={
                    "prompt_preview": pl.Utf8,
                    "response_preview": pl.Utf8,
                    "criteria": pl.Utf8,
                    "score": pl.Float64,
                    "mode": pl.Utf8,
                    "latency_ms": pl.Float64,
                }
            )
        return pl.DataFrame(
            [
                {
                    "prompt_preview": e.prompt[:120],
                    "response_preview": e.response[:120],
                    "criteria": e.criteria,
                    "score": e.verdict.score,
                    "mode": e.verdict.mode,
                    "latency_ms": e.verdict.latency_ms,
                }
                for e in self._log
            ]
        )

    def refusal_df(self) -> pl.DataFrame:
        if not self._refusal_log:
            return pl.DataFrame(
                schema={
                    "label": pl.Utf8,
                    "n": pl.Int64,
                    "refused": pl.Int64,
                    "rate": pl.Float64,
                }
            )
        return pl.DataFrame(self._refusal_log)

    def faithfulness_df(self) -> pl.DataFrame:
        if not self._faithful_log:
            return pl.DataFrame(
                schema={
                    "faithfulness": pl.Float64,
                    "mode": pl.Utf8,
                    "judge_model": pl.Utf8,
                    "latency_ms": pl.Float64,
                }
            )
        return pl.DataFrame(self._faithful_log).drop("rationale", strict=False)

    # ── Plots ──────────────────────────────────────────────────────────

    def plot_output_dashboard(self) -> go.Figure:
        """Score distribution + refusal bars + faithfulness histogram."""
        from plotly.subplots import make_subplots

        fig = make_subplots(
            rows=2,
            cols=2,
            subplot_titles=(
                "Judge scores",
                "Faithfulness",
                "Refusal rate by label",
                "Score by criteria",
            ),
        )

        # (1,1) Judge scores histogram.
        results = self.results_df()
        if results.height:
            fig.add_trace(
                go.Histogram(
                    x=results["score"].to_list(),
                    marker_color=PRIMARY,
                    nbinsx=20,
                ),
                row=1,
                col=1,
            )
        # (1,2) Faithfulness histogram.
        faith = self.faithfulness_df()
        if faith.height:
            fig.add_trace(
                go.Histogram(
                    x=faith["faithfulness"].to_list(),
                    marker_color=WARN,
                    nbinsx=20,
                ),
                row=1,
                col=2,
            )
        # (2,1) Refusal bars.
        ref_df = self.refusal_df()
        if ref_df.height:
            fig.add_trace(
                go.Bar(
                    x=ref_df["label"].to_list(),
                    y=ref_df["rate"].to_list(),
                    marker_color=ACCENT,
                ),
                row=2,
                col=1,
            )
        # (2,2) Score by criteria.
        if results.height:
            agg = (
                results.group_by("criteria")
                .agg(pl.col("score").mean().alias("mean_score"))
                .sort("mean_score")
            )
            fig.add_trace(
                go.Bar(
                    x=agg["criteria"].to_list(),
                    y=agg["mean_score"].to_list(),
                    marker_color=PRIMARY,
                ),
                row=2,
                col=2,
            )

        fig.update_layout(
            title="Output Lens Dashboard (Stethoscope)",
            template=TEMPLATE,
            showlegend=False,
            height=640,
        )
        return fig

    # ── Report ─────────────────────────────────────────────────────────

    def report(self) -> str:
        """Auto-diagnosis in plain text. One line per finding."""
        out: list[str] = []
        results = self.results_df()
        if results.height:
            mean = float(results["score"].mean() or 0.0)
            real_frac = float((results["mode"] == "real").mean() or 0.0)
            out.append(
                f"judge: {results.height} calls, mean score={mean:.2f}, "
                f"real_mode={real_frac:.0%}"
            )
            low = results.filter(pl.col("score") < 0.5).height
            if low:
                out.append(f"  -> {low} calls scored below 0.5 — review rationales")
        faith = self.faithfulness_df()
        if faith.height:
            mean_f = float(faith["faithfulness"].mean() or 0.0)
            out.append(f"faithfulness: {faith.height} calls, mean={mean_f:.2f}")
            if mean_f < 0.7:
                out.append("  -> faithfulness below 0.7 — retrieval or grounding weak")
        ref = self.refusal_df()
        if ref.height:
            for row in ref.iter_rows(named=True):
                out.append(
                    f"refusal[{row['label']}]: {row['rate']:.0%} "
                    f"({row['refused']}/{row['n']})"
                )
        if self._consistency_log:
            last = self._consistency_log[-1]
            out.append(
                f"self-consistency: mean agreement={last['mean_agreement']:.2f}, "
                f"outliers={last['n_outliers']}"
            )
        if not out:
            return "output-lens: no readings recorded yet."
        return "output-lens:\n  " + "\n  ".join(out)


# ════════════════════════════════════════════════════════════════════════
# Helpers
# ════════════════════════════════════════════════════════════════════════


def _looks_like_refusal(text: str) -> bool:
    lower = text.lower()
    return any(marker in lower for marker in _REFUSAL_MARKERS)


def _build_similarity_fn():
    """Return a ``(a, b) -> float`` similarity. ROUGE-L when available, else Jaccard."""
    try:
        from rouge_score import rouge_scorer

        scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

        def _fn(a: str, b: str) -> float:
            return scorer.score(a, b)["rougeL"].fmeasure

        return _fn
    except ImportError:
        logger.info(
            "output.self_consistency.fallback_jaccard",
            extra={"reason": "rouge_score not installed"},
        )

        def _jaccard(a: str, b: str) -> float:
            ta = Counter(a.lower().split())
            tb = Counter(b.lower().split())
            if not ta and not tb:
                return 1.0
            inter = sum((ta & tb).values())
            union = sum((ta | tb).values())
            return inter / union if union else 0.0

        return _jaccard


# ── shared/mlfp06/diagnostics/alignment.py ──
"""Lens 5 — Alignment Diagnostics (the ECG).

Question answered: *Is the fine-tuning signal rewarding the right thing?*

Three primary readings:

    1. **Pair evaluation** — given a ``base_policy`` and a ``tuned_policy``
       plus a preference set, compute KL(base || tuned), reward-margin,
       and pairwise win-rate.
    2. **Training-curve tracking** — pull per-step metrics from a Kailash
       Align :class:`~kailash_align.AlignmentPipeline` (or any iterable of
       ``{step, reward, kl, loss, ...}`` dicts) and record them.
    3. **Reward-hacking detection** — flag the canonical signature of
       sudden reward spike co-occurring with a KL blow-up.

This lens does **no model loading**. It consumes preference tuples,
log-prob arrays, and training metrics — the heavy training runs happen
in the Align framework, this lens observes them.

``trl``'s statistical helpers are used when available (``trl.trainer.utils``
for KL estimators); otherwise the lens falls back to closed-form
implementations (per ``rules/dependencies.md`` — fallback is permitted
because the loud optional-dep error is emitted only by the trl-exclusive
methods).

Quick start::


    align = AlignmentDiagnostics()
    align.evaluate_pair(base_logprobs, tuned_logprobs, preferences)
    align.track_training(pipeline.metrics_stream())
    align.detect_reward_hacking(threshold=2.5)
    align.plot_alignment_dashboard().show()
    print(align.report())
"""

import logging
import math
import uuid
from dataclasses import dataclass
from typing import Any, Iterable, Sequence

import plotly.graph_objects as go
import polars as pl


logger = logging.getLogger(__name__)

__all__ = ["AlignmentDiagnostics"]


# ════════════════════════════════════════════════════════════════════════
# Data classes
# ════════════════════════════════════════════════════════════════════════


@dataclass
class _PairReading:
    label: str
    kl_divergence: float
    reward_margin: float
    win_rate: float
    n: int
    mode: str


@dataclass
class _TrainingStep:
    step: int
    reward: float
    kl: float
    loss: float
    extras: dict[str, Any]


@dataclass
class _HackFinding:
    step: int
    reward_zscore: float
    kl_value: float
    reward_value: float
    label: str


# ════════════════════════════════════════════════════════════════════════
# AlignmentDiagnostics
# ════════════════════════════════════════════════════════════════════════


class AlignmentDiagnostics:
    """Alignment-lens diagnostics — KL, reward margin, win-rate, hacking scan.

    Args:
        label: Short tag applied to every recorded reading (useful when
            comparing multiple runs side-by-side).
    """

    def __init__(self, *, label: str = "run") -> None:
        self._label = label
        self._pair_log: list[_PairReading] = []
        self._training_log: list[_TrainingStep] = []
        self._hack_findings: list[_HackFinding] = []
        logger.info("alignment_diagnostics.init", extra={"label": label})

    def __enter__(self) -> "AlignmentDiagnostics":
        return self

    def __exit__(self, *exc: Any) -> None:
        pass

    # ── Pair evaluation ────────────────────────────────────────────────

    def evaluate_pair(
        self,
        base_policy: Sequence[Sequence[float]],
        tuned_policy: Sequence[Sequence[float]],
        preferences: Sequence[dict[str, Any]],
        *,
        label: str | None = None,
        run_id: str | None = None,
    ) -> pl.DataFrame:
        """Compute KL(base || tuned), reward margin, and pair win-rate.

        Args:
            base_policy: Per-example token log-probabilities from the base
                model. Length ``N``, each element a sequence of log-probs.
            tuned_policy: Same shape, from the tuned model.
            preferences: Iterable of ``{chosen_reward, rejected_reward,
                chosen_won}`` dicts. ``chosen_won`` is a bool.
            label: Optional sub-label overriding the instance label.
            run_id: Correlation ID per ``rules/observability.md``.

        Returns:
            A one-row Polars DataFrame.
        """
        if len(base_policy) != len(tuned_policy):
            raise ValueError("base_policy and tuned_policy must be same length")
        run_id = run_id or f"align_pair-{uuid.uuid4().hex[:12]}"
        label = label or self._label

        kls = [_kl_from_logprobs(b, t) for b, t in zip(base_policy, tuned_policy)]
        kl_mean = _mean(kls)

        if preferences:
            margins = [
                float(p["chosen_reward"]) - float(p["rejected_reward"])
                for p in preferences
            ]
            wins = sum(1 for p in preferences if bool(p.get("chosen_won")))
            win_rate = wins / len(preferences)
            reward_margin = _mean(margins)
        else:
            win_rate = float("nan")
            reward_margin = float("nan")

        reading = _PairReading(
            label=label,
            kl_divergence=kl_mean,
            reward_margin=reward_margin,
            win_rate=win_rate,
            n=len(base_policy),
            mode="real",
        )
        self._pair_log.append(reading)
        logger.info(
            "alignment.evaluate_pair",
            extra={
                "run_id": run_id,
                "label": label,
                "kl": kl_mean,
                "reward_margin": reward_margin,
                "win_rate": win_rate,
                "n": len(base_policy),
                "mode": "real",
            },
        )
        return pl.DataFrame(
            [
                {
                    "label": reading.label,
                    "kl_divergence": reading.kl_divergence,
                    "reward_margin": reading.reward_margin,
                    "win_rate": reading.win_rate,
                    "n": reading.n,
                }
            ]
        )

    def kl_divergence(
        self,
        p_logprobs: Sequence[float],
        q_logprobs: Sequence[float],
    ) -> float:
        """KL(p || q) from token log-probabilities.

        When ``trl`` is installed and exposes
        ``trl.trainer.utils.kl_divergence``, that implementation is used;
        otherwise a closed-form estimator runs locally.
        """
        try:
            from trl.trainer.utils import kl_divergence as trl_kl  # type: ignore[import-not-found]
        except ImportError:
            return _kl_from_logprobs(p_logprobs, q_logprobs)
        try:
            import torch  # type: ignore[import-not-found]

            p = torch.tensor(list(p_logprobs))
            q = torch.tensor(list(q_logprobs))
            return float(trl_kl(p, q).mean().item())
        except Exception as exc:  # pragma: no cover — defensive
            logger.warning(
                "alignment.trl_kl_error",
                extra={"error": str(exc), "mode": "real"},
            )
            return _kl_from_logprobs(p_logprobs, q_logprobs)

    def win_rate(self, preferences: Sequence[dict[str, Any]]) -> float:
        """Fraction of preference rows where the chosen policy won."""
        if not preferences:
            return float("nan")
        wins = sum(1 for p in preferences if bool(p.get("chosen_won")))
        return wins / len(preferences)

    # ── Training-curve tracking ────────────────────────────────────────

    def track_training(
        self,
        metrics: Iterable[dict[str, Any]] | Any,
        *,
        run_id: str | None = None,
    ) -> pl.DataFrame:
        """Record a training-metrics stream from an Align pipeline.

        Accepts either:

            * an iterable of ``{step, reward, kl, loss, ...}`` dicts, or
            * a Kailash Align ``AlignmentPipeline`` exposing a
              ``metrics_stream()`` / ``.metrics`` attribute.

        Any missing field is defaulted to ``float("nan")`` so partial
        streams (e.g. SFT runs without a reward signal) still produce a
        usable DataFrame.
        """
        run_id = run_id or f"align_track-{uuid.uuid4().hex[:12]}"
        iterable = _resolve_metrics_iterable(metrics)
        rows: list[dict[str, Any]] = []
        for raw in iterable:
            step = int(raw.get("step", len(self._training_log)))
            reward = float(raw.get("reward", float("nan")))
            kl = float(raw.get("kl", raw.get("kl_divergence", float("nan"))))
            loss = float(raw.get("loss", float("nan")))
            extras = {
                k: v
                for k, v in raw.items()
                if k not in {"step", "reward", "kl", "kl_divergence", "loss"}
            }
            self._training_log.append(
                _TrainingStep(step=step, reward=reward, kl=kl, loss=loss, extras=extras)
            )
            rows.append({"step": step, "reward": reward, "kl": kl, "loss": loss})
        df = pl.DataFrame(rows) if rows else _empty_training_df()
        logger.info(
            "alignment.track_training",
            extra={
                "run_id": run_id,
                "steps": df.height,
                "source": "align_pipeline",
                "mode": "real",
            },
        )
        return df

    # ── Reward hacking detection ───────────────────────────────────────

    def detect_reward_hacking(
        self,
        history: Sequence[dict[str, Any]] | None = None,
        *,
        threshold: float = 2.5,
        label: str | None = None,
    ) -> pl.DataFrame:
        """Flag reward-spike + KL-blowup steps.

        Reward-hacking's canonical signature is a sudden jump in reward
        that coincides with a divergence blow-up — the model has learned
        a shortcut the reward model rewards but the base distribution
        doesn't support.

        Args:
            history: Optional pre-recorded training history. When
                ``None``, uses the history accumulated via
                :meth:`track_training`.
            threshold: Z-score above which a reward delta is flagged.
            label: Optional label applied to findings.

        Returns:
            DataFrame of findings (``step``, ``reward_zscore``,
            ``kl_value``, ``reward_value``).
        """
        if history is not None:
            series = [
                _TrainingStep(
                    step=int(h.get("step", i)),
                    reward=float(h.get("reward", float("nan"))),
                    kl=float(h.get("kl", float("nan"))),
                    loss=float(h.get("loss", float("nan"))),
                    extras={},
                )
                for i, h in enumerate(history)
            ]
        else:
            series = list(self._training_log)
        if len(series) < 4:
            return _empty_findings_df()

        rewards = [s.reward for s in series if not math.isnan(s.reward)]
        if len(rewards) < 4:
            return _empty_findings_df()
        mu = _mean(rewards)
        sigma = _stdev(rewards, mu) or 1e-9
        median_kl = _median([s.kl for s in series if not math.isnan(s.kl)])

        label = label or self._label
        findings: list[_HackFinding] = []
        for prev, cur in zip(series, series[1:]):
            if math.isnan(cur.reward) or math.isnan(prev.reward):
                continue
            delta = cur.reward - prev.reward
            z = delta / sigma
            if (
                z > threshold
                and not math.isnan(cur.kl)
                and cur.kl > max(median_kl * 1.5, 0.05)
            ):
                findings.append(
                    _HackFinding(
                        step=cur.step,
                        reward_zscore=z,
                        kl_value=cur.kl,
                        reward_value=cur.reward,
                        label=label,
                    )
                )

        self._hack_findings.extend(findings)
        if findings:
            logger.warning(
                "alignment.reward_hacking.detected",
                extra={
                    "label": label,
                    "n_findings": len(findings),
                    "threshold_z": threshold,
                },
            )
        return (
            pl.DataFrame(
                [
                    {
                        "step": f.step,
                        "reward_zscore": f.reward_zscore,
                        "kl_value": f.kl_value,
                        "reward_value": f.reward_value,
                        "label": f.label,
                    }
                    for f in findings
                ]
            )
            if findings
            else _empty_findings_df()
        )

    # ── DataFrames ─────────────────────────────────────────────────────

    def pair_df(self) -> pl.DataFrame:
        if not self._pair_log:
            return pl.DataFrame(
                schema={
                    "label": pl.Utf8,
                    "kl_divergence": pl.Float64,
                    "reward_margin": pl.Float64,
                    "win_rate": pl.Float64,
                    "n": pl.Int64,
                }
            )
        return pl.DataFrame(
            [
                {
                    "label": r.label,
                    "kl_divergence": r.kl_divergence,
                    "reward_margin": r.reward_margin,
                    "win_rate": r.win_rate,
                    "n": r.n,
                }
                for r in self._pair_log
            ]
        )

    def training_df(self) -> pl.DataFrame:
        if not self._training_log:
            return _empty_training_df()
        return pl.DataFrame(
            [
                {"step": s.step, "reward": s.reward, "kl": s.kl, "loss": s.loss}
                for s in self._training_log
            ]
        )

    def findings_df(self) -> pl.DataFrame:
        if not self._hack_findings:
            return _empty_findings_df()
        return pl.DataFrame(
            [
                {
                    "step": f.step,
                    "reward_zscore": f.reward_zscore,
                    "kl_value": f.kl_value,
                    "reward_value": f.reward_value,
                    "label": f.label,
                }
                for f in self._hack_findings
            ]
        )

    # ── Plots ──────────────────────────────────────────────────────────

    def plot_alignment_dashboard(self) -> go.Figure:
        """Reward curve, KL curve, win-rate bars; hacking findings highlighted."""
        if not self._training_log and not self._pair_log:
            return empty_figure("Alignment Lens Dashboard (ECG)")

        from plotly.subplots import make_subplots

        fig = make_subplots(
            rows=2,
            cols=2,
            subplot_titles=(
                "Reward over training steps",
                "KL divergence over training steps",
                "Pair win-rate",
                "Reward vs KL (hacking scan)",
            ),
        )

        train_df = self.training_df()
        if train_df.height:
            fig.add_trace(
                go.Scatter(
                    x=train_df["step"].to_list(),
                    y=train_df["reward"].to_list(),
                    mode="lines+markers",
                    line=dict(color=PRIMARY),
                    name="reward",
                ),
                row=1,
                col=1,
            )
            fig.add_trace(
                go.Scatter(
                    x=train_df["step"].to_list(),
                    y=train_df["kl"].to_list(),
                    mode="lines+markers",
                    line=dict(color=WARN),
                    name="kl",
                ),
                row=1,
                col=2,
            )
            # Scatter colored by z-score flag.
            fig.add_trace(
                go.Scatter(
                    x=train_df["kl"].to_list(),
                    y=train_df["reward"].to_list(),
                    mode="markers",
                    marker=dict(
                        color=PRIMARY,
                        size=8,
                        line=dict(width=1, color=MUTED),
                    ),
                    name="steps",
                ),
                row=2,
                col=2,
            )
            findings_df = self.findings_df()
            if findings_df.height:
                fig.add_trace(
                    go.Scatter(
                        x=findings_df["kl_value"].to_list(),
                        y=findings_df["reward_value"].to_list(),
                        mode="markers",
                        marker=dict(
                            color=WARN, size=14, symbol="x", line=dict(width=2)
                        ),
                        name="hack flag",
                    ),
                    row=2,
                    col=2,
                )

        pair_df = self.pair_df()
        if pair_df.height:
            fig.add_trace(
                go.Bar(
                    x=pair_df["label"].to_list(),
                    y=pair_df["win_rate"].to_list(),
                    marker_color=PRIMARY,
                    name="win_rate",
                ),
                row=2,
                col=1,
            )

        fig.update_layout(
            title="Alignment Lens Dashboard (ECG)",
            template=TEMPLATE,
            showlegend=False,
            height=640,
        )
        return fig

    # ── Report ─────────────────────────────────────────────────────────

    def report(self) -> str:
        """Plain-text Prescription Pad for the alignment lens."""
        out: list[str] = []
        pair_df = self.pair_df()
        if pair_df.height:
            for row in pair_df.iter_rows(named=True):
                out.append(
                    f"pair[{row['label']}]: KL={row['kl_divergence']:.3f}, "
                    f"margin={row['reward_margin']:.3f}, "
                    f"win_rate={row['win_rate']:.0%} (n={row['n']})"
                )
                if row["kl_divergence"] > 1.0:
                    out.append("  -> KL above 1.0 — tuned model drifted far from base")
                if row["reward_margin"] < 0.05:
                    out.append("  -> margin below 0.05 — preference signal is too weak")

        train_df = self.training_df()
        if train_df.height:
            mean_r = float(train_df["reward"].mean() or 0.0)
            mean_kl = float(train_df["kl"].mean() or 0.0)
            out.append(
                f"training: {train_df.height} steps, mean reward={mean_r:.3f}, "
                f"mean KL={mean_kl:.3f}"
            )

        findings_df = self.findings_df()
        if findings_df.height:
            out.append(
                f"hacking scan: {findings_df.height} suspected step(s) — "
                f"reward spike + KL blow-up"
            )
            top = findings_df.row(0, named=True)
            out.append(
                f"  -> step {top['step']}: z={top['reward_zscore']:.2f}, "
                f"kl={top['kl_value']:.3f}"
            )

        if not out:
            return "alignment-lens: no readings recorded yet."
        return "alignment-lens:\n  " + "\n  ".join(out)


# ════════════════════════════════════════════════════════════════════════
# Helpers
# ════════════════════════════════════════════════════════════════════════


def _mean(xs: Sequence[float]) -> float:
    xs = [x for x in xs if not math.isnan(x)]
    return sum(xs) / len(xs) if xs else float("nan")


def _stdev(xs: Sequence[float], mu: float) -> float:
    xs = [x for x in xs if not math.isnan(x)]
    if len(xs) < 2:
        return 0.0
    return math.sqrt(sum((x - mu) ** 2 for x in xs) / (len(xs) - 1))


def _median(xs: Sequence[float]) -> float:
    xs = sorted(x for x in xs if not math.isnan(x))
    if not xs:
        return 0.0
    mid = len(xs) // 2
    if len(xs) % 2:
        return xs[mid]
    return 0.5 * (xs[mid - 1] + xs[mid])


def _kl_from_logprobs(
    p_logprobs: Sequence[float], q_logprobs: Sequence[float]
) -> float:
    """KL(P || Q) estimator from paired token log-probabilities.

    Given log p(x_t) and log q(x_t) for the same sequence, the Monte
    Carlo estimator is E_{x ~ P}[log p - log q] ≈ mean(p_logprobs -
    q_logprobs). We also clip the ratio to avoid inf.
    """
    n = min(len(p_logprobs), len(q_logprobs))
    if n == 0:
        return 0.0
    diffs = []
    for i in range(n):
        d = float(p_logprobs[i]) - float(q_logprobs[i])
        # Clip extreme values so a single bad token can't dominate.
        d = max(-50.0, min(50.0, d))
        diffs.append(d)
    return sum(diffs) / n


def _resolve_metrics_iterable(obj: Any) -> Iterable[dict[str, Any]]:
    """Extract a metrics iterable from an ``AlignmentPipeline`` or a list."""
    if obj is None:
        return []
    if hasattr(obj, "metrics_stream"):
        return obj.metrics_stream()
    if hasattr(obj, "metrics"):
        return list(obj.metrics)
    return obj  # assume iterable of dicts


def _empty_training_df() -> pl.DataFrame:
    return pl.DataFrame(
        schema={
            "step": pl.Int64,
            "reward": pl.Float64,
            "kl": pl.Float64,
            "loss": pl.Float64,
        }
    )


def _empty_findings_df() -> pl.DataFrame:
    return pl.DataFrame(
        schema={
            "step": pl.Int64,
            "reward_zscore": pl.Float64,
            "kl_value": pl.Float64,
            "reward_value": pl.Float64,
            "label": pl.Utf8,
        }
    )


# ── shared/mlfp06/diagnostics/interpretability.py ──
"""Lens 2 — Interpretability (the X-Ray).

Question answered: *What does the model attend to, and what circuit produces the answer?*

Works with open-weight models only (Llama, Gemma, Phi, Mistral). API-only
models (GPT, Claude, Gemini) explicitly return ``{"mode": "not_applicable"}``
— the lens is honest about what it cannot do, per rules/zero-tolerance.md
Rule 2 (no fake readings).

Default model: ``google/gemma-2-2b`` (Gemma Scope SAE coverage for
:meth:`sae_features`). Loading happens lazily on first use so the class
is cheap to import.

The underlying libraries are ``transformer_lens`` (attention + activation
hooks), ``sae_lens`` (pre-trained Gemma Scope SAEs), and optional
``nnterp`` (unified 2025 interface). Students read the output, they do
not train the SAEs — per design doc §11 Non-goals.
"""

import logging
import os
import uuid
from typing import Any, Sequence

import plotly.graph_objects as go
import polars as pl


logger = logging.getLogger(__name__)

__all__ = ["InterpretabilityDiagnostics"]

DEFAULT_MODEL = "google/gemma-2-2b"

# Models we know the lens cannot introspect (API-only).
_API_ONLY_PREFIXES: tuple[str, ...] = (
    "gpt-",
    "o1-",
    "o3-",
    "o4-",
    "claude-",
    "gemini-",
    "deepseek-",  # API tier
)


class InterpretabilityDiagnostics:
    """X-ray lens — attention heatmaps, logit-lens, probes, SAE features.

    Args:
        model_name: HuggingFace model identifier. Defaults to
            ``google/gemma-2-2b`` (Gemma Scope SAE coverage).
        device: Torch device string (``"cuda"``, ``"mps"``, ``"cpu"``).
            Auto-detected when ``None``.
        dtype: Torch dtype for weights — ``"float16"`` saves VRAM.
        run_id_prefix: Prefix for auto-generated correlation IDs.
    """

    def __init__(
        self,
        *,
        model_name: str = DEFAULT_MODEL,
        device: str | None = None,
        dtype: str = "float16",
        run_id_prefix: str = "attn",
    ) -> None:
        self.model_name = model_name
        self._device = device
        self._dtype = dtype
        self._run_id_prefix = run_id_prefix
        self._model: Any = None
        self._attention_log: list[dict[str, Any]] = []
        self._logit_log: list[dict[str, Any]] = []
        self._sae_log: list[dict[str, Any]] = []
        self._probe_log: list[dict[str, Any]] = []
        logger.info(
            "interp_diagnostics.init",
            extra={"model": model_name, "device": device or "auto", "dtype": dtype},
        )

    def __enter__(self) -> "InterpretabilityDiagnostics":
        return self

    def __exit__(self, *exc: Any) -> None:
        self.close()

    def close(self) -> None:
        """Release the HookedTransformer + clear caches."""
        self._model = None
        try:
            import torch

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass

    # ── Applicability check ────────────────────────────────────────────

    def _is_api_only(self, model: str | None = None) -> bool:
        m = (model or self.model_name).lower()
        return any(m.startswith(p) for p in _API_ONLY_PREFIXES)

    def _not_applicable(self, method: str, model: str | None = None) -> dict[str, Any]:
        logger.info(
            "interp.not_applicable",
            extra={"method": method, "model": model or self.model_name, "mode": "real"},
        )
        return {
            "mode": "not_applicable",
            "method": method,
            "model": model or self.model_name,
            "reason": (
                "attention lens requires open-weight models (Llama, Gemma, Phi, "
                "Mistral). API-only models (GPT, Claude, Gemini) cannot be X-rayed."
            ),
        }

    # ── Model loading ─────────────────────────────────────────────────

    def _load_model(self) -> Any:
        """Lazy-load the HookedTransformer. First call only."""
        if self._model is not None:
            return self._model
        if self._is_api_only():
            raise RuntimeError(
                f"{self.model_name} is API-only; cannot load weights. Use an "
                "open-weight model such as google/gemma-2-2b or meta-llama/Llama-3.2-1B."
            )
        try:
            from transformer_lens import HookedTransformer
        except ImportError as exc:  # pragma: no cover — optional extra
            raise ImportError(
                "InterpretabilityDiagnostics requires transformer_lens. "
                "Install with `pip install transformer-lens`."
            ) from exc

        device = self._device or _auto_device()
        # HuggingFace token for gated Gemma/Llama weights.
        token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
        load_kwargs: dict[str, Any] = {"device": device}
        if self._dtype == "float16":
            import torch

            load_kwargs["torch_dtype"] = torch.float16
        if token:
            load_kwargs["token"] = token

        logger.info(
            "interp.load_model.start",
            extra={"model": self.model_name, "device": device, "dtype": self._dtype},
        )
        self._model = HookedTransformer.from_pretrained(self.model_name, **load_kwargs)
        logger.info(
            "interp.load_model.ok",
            extra={"model": self.model_name, "n_layers": self._model.cfg.n_layers},
        )
        return self._model

    # ── Attention heatmap ──────────────────────────────────────────────

    def attention_heatmap(
        self,
        prompt: str,
        *,
        layer: int = 0,
        head: int = 0,
        run_id: str | None = None,
    ) -> go.Figure:
        """Token-to-token attention weights at (layer, head) as a heatmap.

        Returns a Plotly Figure; also records the (tokens, matrix) reading
        for later dashboard aggregation.
        """
        if self._is_api_only():
            # For API-only models the plot is a labelled empty figure.
            _ = self._not_applicable("attention_heatmap")
            return empty_figure(
                f"Attention Heatmap — layer {layer}, head {head}",
                note="not applicable for API-only models",
            )

        run_id = run_id or f"{self._run_id_prefix}-{uuid.uuid4().hex[:12]}"
        model = self._load_model()
        import torch

        tokens = model.to_tokens(prompt)
        with torch.no_grad():
            _, cache = model.run_with_cache(tokens)
        # transformer_lens stores attention as [batch, head, query, key]
        attn = cache["pattern", layer][0, head].to("cpu").float().numpy()
        labels = [model.to_string(t) for t in tokens[0]]
        labels = [lbl.replace("\n", "\\n") or "∅" for lbl in labels]

        fig = go.Figure(
            go.Heatmap(
                z=attn,
                x=labels,
                y=labels,
                colorscale="Viridis",
                colorbar=dict(title="attention"),
            )
        )
        fig.update_layout(
            title=f"Attention — {self.model_name} · L{layer} H{head}",
            xaxis_title="key token",
            yaxis_title="query token",
            template=TEMPLATE,
            height=600,
        )
        self._attention_log.append(
            {
                "run_id": run_id,
                "layer": layer,
                "head": head,
                "n_tokens": len(labels),
                "mode": "real",
            }
        )
        logger.info(
            "interp.attention_heatmap.ok",
            extra={
                "run_id": run_id,
                "layer": layer,
                "head": head,
                "n_tokens": len(labels),
                "mode": "real",
            },
        )
        return fig

    # ── Logit lens ─────────────────────────────────────────────────────

    def logit_lens(
        self,
        prompt: str,
        *,
        top_k: int = 5,
        run_id: str | None = None,
    ) -> pl.DataFrame:
        """Early-exit predictions per layer.

        Projects each layer's residual stream through the unembedding and
        records the top-``k`` tokens + probabilities. Returns a Polars
        DataFrame with columns ``layer``, ``rank``, ``token``, ``prob``.

        On API-only models, returns an empty DataFrame tagged
        ``mode="not_applicable"`` instead of raising.
        """
        if self._is_api_only():
            reading = self._not_applicable("logit_lens")
            self._logit_log.append(reading)
            return pl.DataFrame(
                schema={
                    "layer": pl.Int64,
                    "rank": pl.Int64,
                    "token": pl.Utf8,
                    "prob": pl.Float64,
                    "mode": pl.Utf8,
                }
            )

        run_id = run_id or f"{self._run_id_prefix}-ll-{uuid.uuid4().hex[:12]}"
        model = self._load_model()
        import torch

        tokens = model.to_tokens(prompt)
        with torch.no_grad():
            _, cache = model.run_with_cache(tokens, remove_batch_dim=True)

        rows: list[dict[str, Any]] = []
        last_tok = tokens.shape[-1] - 1
        for layer in range(model.cfg.n_layers):
            resid = cache["resid_post", layer][last_tok]
            normed = model.ln_final(resid)
            logits = model.unembed(normed)
            probs = torch.softmax(logits, dim=-1)
            top = torch.topk(probs, k=top_k)
            for rank, (p, tok_id) in enumerate(
                zip(top.values.tolist(), top.indices.tolist())
            ):
                rows.append(
                    {
                        "layer": layer,
                        "rank": rank,
                        "token": model.to_string(tok_id).replace("\n", "\\n"),
                        "prob": float(p),
                        "mode": "real",
                    }
                )
        df = pl.DataFrame(rows)
        self._logit_log.append(
            {
                "run_id": run_id,
                "n_layers": model.cfg.n_layers,
                "top_k": top_k,
                "mode": "real",
            }
        )
        logger.info(
            "interp.logit_lens.ok",
            extra={
                "run_id": run_id,
                "n_layers": model.cfg.n_layers,
                "top_k": top_k,
                "mode": "real",
            },
        )
        return df

    def plot_logit_lens(self, df: pl.DataFrame) -> go.Figure:
        """Heatmap of top-1 logit-lens probability per layer."""
        if df.height == 0:
            return empty_figure("Logit Lens", note="no data or not applicable")
        top1 = df.filter(pl.col("rank") == 0).sort("layer")
        fig = go.Figure(
            go.Bar(
                x=top1["layer"].to_list(),
                y=top1["prob"].to_list(),
                text=top1["token"].to_list(),
                marker_color=PRIMARY,
            )
        )
        style(
            fig,
            f"Logit Lens — top-1 probability per layer ({self.model_name})",
            x="layer",
            y="probability",
        )
        return fig

    # ── Linear probe ───────────────────────────────────────────────────

    def probe(
        self,
        prompts: Sequence[str],
        labels: Sequence[int],
        *,
        layer: int,
        run_id: str | None = None,
    ) -> dict[str, Any]:
        """Train a linear probe on layer activations.

        The caller supplies ``prompts`` and corresponding integer ``labels``.
        The method extracts the last-token residual stream at ``layer``,
        fits a logistic regression (scikit-learn), and reports CV accuracy.
        """
        if self._is_api_only():
            return self._not_applicable("probe")

        if len(prompts) != len(labels):
            raise ValueError("prompts and labels must be same length")
        if len(set(labels)) < 2:
            raise ValueError("probe needs at least 2 distinct labels")

        run_id = run_id or f"{self._run_id_prefix}-probe-{uuid.uuid4().hex[:12]}"
        model = self._load_model()
        import numpy as np
        import torch
        from sklearn.linear_model import LogisticRegression
        from sklearn.model_selection import cross_val_score

        features: list[np.ndarray] = []
        for p in prompts:
            tokens = model.to_tokens(p)
            with torch.no_grad():
                _, cache = model.run_with_cache(tokens, remove_batch_dim=True)
            last = cache["resid_post", layer][-1].to("cpu").float().numpy()
            features.append(last)
        X = np.stack(features)
        y = np.asarray(labels)
        clf = LogisticRegression(max_iter=500)
        scores = cross_val_score(clf, X, y, cv=min(5, len(set(labels))))
        acc = float(scores.mean())
        row = {
            "run_id": run_id,
            "layer": layer,
            "n_prompts": len(prompts),
            "n_classes": len(set(labels)),
            "cv_accuracy": acc,
            "mode": "real",
        }
        self._probe_log.append(row)
        logger.info("interp.probe.ok", extra=row)
        return row

    # ── SAE features (Gemma Scope) ─────────────────────────────────────

    def sae_features(
        self,
        prompt: str,
        *,
        layer: int,
        top_k: int = 10,
        release: str | None = None,
        run_id: str | None = None,
    ) -> pl.DataFrame:
        """Load a pre-trained SAE and return the top-``k`` active features.

        ``release`` is the :mod:`sae_lens` release identifier. When ``None``,
        the default Gemma Scope release matching ``self.model_name`` is used
        (``gemma-scope-2b-pt-res`` for gemma-2-2b).

        Returns a Polars DataFrame with ``feature_index``, ``activation``,
        and a ``mode`` column. Students do NOT train SAEs (design §11).
        """
        if self._is_api_only():
            reading = self._not_applicable("sae_features")
            self._sae_log.append(reading)
            return pl.DataFrame(
                schema={
                    "feature_index": pl.Int64,
                    "activation": pl.Float64,
                    "mode": pl.Utf8,
                }
            )

        try:
            from sae_lens import SAE
        except ImportError as exc:  # pragma: no cover
            raise ImportError(
                "sae_features requires sae-lens. Install with `pip install sae-lens`."
            ) from exc

        run_id = run_id or f"{self._run_id_prefix}-sae-{uuid.uuid4().hex[:12]}"
        model = self._load_model()
        import torch

        release = release or _default_sae_release(self.model_name)
        sae_id = f"layer_{layer}/width_16k/canonical"
        logger.info(
            "interp.sae_load.start",
            extra={"release": release, "sae_id": sae_id, "layer": layer},
        )
        sae, _cfg_dict, _sparsity = SAE.from_pretrained(release=release, sae_id=sae_id)
        sae = sae.to(model.cfg.device)

        tokens = model.to_tokens(prompt)
        with torch.no_grad():
            _, cache = model.run_with_cache(tokens, remove_batch_dim=True)
            hook_name = f"blocks.{layer}.hook_resid_post"
            resid = cache[hook_name][-1]
            acts = sae.encode(resid.to(sae.device))
        top = torch.topk(acts, k=min(top_k, acts.numel()))
        rows = [
            {
                "feature_index": int(idx),
                "activation": float(val),
                "mode": "real",
            }
            for idx, val in zip(top.indices.tolist(), top.values.tolist())
        ]
        df = pl.DataFrame(rows)
        self._sae_log.append(
            {
                "run_id": run_id,
                "layer": layer,
                "release": release,
                "top_k": top_k,
                "mode": "real",
            }
        )
        logger.info(
            "interp.sae_features.ok",
            extra={
                "run_id": run_id,
                "layer": layer,
                "top_k": top_k,
                "mode": "real",
            },
        )
        return df

    # ── Report ─────────────────────────────────────────────────────────

    def report(self) -> str:
        if self._is_api_only():
            return (
                f"interp-lens: {self.model_name} is API-only — attention, logit "
                "lens, probe, and SAE features are NOT APPLICABLE. Load an "
                "open-weight model (e.g. google/gemma-2-2b)."
            )
        parts: list[str] = []
        if self._attention_log:
            parts.append(f"attention: {len(self._attention_log)} heatmaps recorded")
        if self._logit_log:
            parts.append(f"logit_lens: {len(self._logit_log)} sweeps recorded")
        if self._probe_log:
            last = self._probe_log[-1]
            parts.append(
                f"probe: last CV accuracy={last['cv_accuracy']:.2f} on layer {last['layer']}"
            )
        if self._sae_log:
            parts.append(f"sae: {len(self._sae_log)} feature reads recorded")
        if not parts:
            return "interp-lens: no readings recorded yet."
        return "interp-lens:\n  " + "\n  ".join(parts)


# ════════════════════════════════════════════════════════════════════════
# Helpers
# ════════════════════════════════════════════════════════════════════════


def _auto_device() -> str:
    try:
        import torch

        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return "mps"
        if torch.cuda.is_available():
            return "cuda"
    except Exception:
        pass
    return "cpu"


def _default_sae_release(model_name: str) -> str:
    name = model_name.lower()
    if "gemma-2-2b" in name:
        return "gemma-scope-2b-pt-res"
    if "gemma-2-9b" in name:
        return "gemma-scope-9b-pt-res"
    # Fall back to gemma-2-2b's release — Gemma Scope has the best coverage.
    return "gemma-scope-2b-pt-res"


# ── shared/mlfp06/diagnostics/governance.py ──
"""Lens 6 — Governance Diagnostics (the Flight Recorder).

Question answered: *Is the system operating inside its envelope, and is
the audit chain intact?*

Wraps a PACT ``GovernanceEngine`` (or the ``audit`` facet of a Kaizen
``GovernedSupervisor``). This lens is NEVER responsible for making
governance decisions — it only *reads* what the engine has already
recorded. Per ``rules/framework-first.md`` MANDATORY: policy / envelope
construction is the engine's job, not ours.

Typical use::


    diag = GovernanceDiagnostics(governance=supervisor.audit)
    audit_df = diag.audit_snapshot(last_n=200)
    chain_ok = diag.verify_chain(audit_df)
    budget_df = diag.budget_consumption()
    diag.plot_governance_dashboard().show()
    print(diag.report())
"""

import hashlib
import json
import logging
from dataclasses import dataclass
from typing import Any, Iterable, Sequence

import plotly.graph_objects as go
import polars as pl


logger = logging.getLogger(__name__)

__all__ = ["GovernanceDiagnostics"]


# ════════════════════════════════════════════════════════════════════════
# Internal row shape (what we harvest from the engine)
# ════════════════════════════════════════════════════════════════════════


@dataclass
class _DrillResult:
    scenario: str
    verdict: str
    reason: str
    mode: str


# Canonical envelope dimensions — matches PACT's envelope taxonomy.
_ENVELOPE_DIMENSIONS: tuple[str, ...] = (
    "financial",
    "temporal",
    "data_access",
    "communication",
    "operational",
)

_VERDICT_COLORS = {
    "allow": PRIMARY,
    "warn": ACCENT,
    "block": WARN,
    "escalate": "mediumpurple",
    "unknown": MUTED,
}


class GovernanceDiagnostics:
    """Governance-lens diagnostics — audit snapshot, chain verification, drills.

    Args:
        governance: The governance engine under observation. Accepts:

            * PACT ``GovernanceEngine`` (``.verify_action`` + audit log)
            * Kaizen ``GovernedSupervisor.audit`` facet
            * Any object exposing ``audit_entries`` / ``to_list`` or a
              callable ``verify_action``.

            When ``None``, every data method raises a loud error (per
            ``rules/observability.md`` §3 — no silent fake verdicts).
        run_id: Optional correlation ID.
    """

    def __init__(
        self,
        *,
        governance: Any = None,
        run_id: str | None = None,
    ) -> None:
        self._engine = governance
        self._run_id = run_id
        self._drill_results: list[_DrillResult] = []
        logger.info(
            "governance_diagnostics.init",
            extra={
                "engine_type": type(governance).__name__ if governance else "none",
                "run_id": run_id,
            },
        )

    def __enter__(self) -> "GovernanceDiagnostics":
        return self

    def __exit__(self, *exc: Any) -> None:
        self.close()

    def close(self) -> None:
        """Release any engine handles (best-effort)."""
        # The engine itself is owned by the caller — do not close it.
        self._drill_results = []

    # ── Helpers ────────────────────────────────────────────────────────

    def _require_engine(self, op: str) -> Any:
        if self._engine is None:
            raise RuntimeError(
                f"GovernanceDiagnostics.{op}() requires a governance engine. "
                "Pass governance=PACT GovernanceEngine or "
                "governance=supervisor.audit to the constructor. Silent "
                "degradation to fake verdicts is BLOCKED per "
                "rules/observability.md §3."
            )
        return self._engine

    def _extract_audit_rows(self, engine: Any) -> list[dict[str, Any]]:
        """Normalise whatever the engine exposes into a flat list of dicts.

        Recognised shapes (checked in order):

            * ``engine.audit_entries`` (PACT GovernanceEngine canonical)
            * ``engine.to_list()`` (Kaizen audit facet)
            * ``engine.entries`` (generic list)
            * iterable of dicts / dataclasses
        """
        source: Iterable[Any]
        if hasattr(engine, "audit_entries"):
            source = (
                engine.audit_entries()
                if callable(engine.audit_entries)
                else engine.audit_entries
            )
        elif hasattr(engine, "to_list"):
            source = engine.to_list()
        elif hasattr(engine, "entries"):
            source = engine.entries
        elif isinstance(engine, (list, tuple)):
            source = engine
        else:
            raise TypeError(
                f"GovernanceDiagnostics cannot read audit log from "
                f"{type(engine).__name__}. Expected .audit_entries, "
                f".to_list(), .entries, or a list."
            )

        rows: list[dict[str, Any]] = []
        for raw in source:
            if isinstance(raw, dict):
                rows.append(raw)
            elif hasattr(raw, "__dict__"):
                # Dataclass or plain object.
                rows.append(
                    {k: v for k, v in vars(raw).items() if not k.startswith("_")}
                )
            else:
                logger.warning(
                    "governance.audit.skip_unknown_row",
                    extra={"row_type": type(raw).__name__},
                )
        return rows

    # ── Audit snapshot ─────────────────────────────────────────────────

    def audit_snapshot(self, last_n: int = 100) -> pl.DataFrame:
        """Return the last ``last_n`` envelope decisions as a Polars frame.

        Columns (best-effort, missing fields become nulls):

            * ``timestamp`` — ISO-8601 str or float epoch, whatever the
              engine records.
            * ``subject`` — Director / Tenant / Role (PACT D/T/R).
            * ``action`` — The proposed action string.
            * ``verdict`` — ``allow`` / ``warn`` / ``block`` / ``escalate``.
            * ``reason`` — Short rationale from the engine.
            * ``hash`` / ``prev_hash`` — If the engine records them.
        """
        engine = self._require_engine("audit_snapshot")
        rows = self._extract_audit_rows(engine)
        if last_n > 0 and len(rows) > last_n:
            rows = rows[-last_n:]

        # Canonicalise the key names we care about. Accept several aliases.
        canonical: list[dict[str, Any]] = []
        for r in rows:
            canonical.append(
                {
                    "timestamp": r.get("timestamp") or r.get("ts") or r.get("time"),
                    "subject": r.get("subject") or r.get("director") or r.get("actor"),
                    "action": r.get("action") or r.get("operation"),
                    "verdict": (r.get("verdict") or r.get("decision") or "unknown"),
                    "reason": r.get("reason") or r.get("rationale") or "",
                    "hash": r.get("hash"),
                    "prev_hash": r.get("prev_hash"),
                }
            )

        logger.info(
            "governance.audit_snapshot",
            extra={
                "run_id": self._run_id,
                "rows": len(canonical),
                "source": "governance_engine",
                "mode": "real",
            },
        )
        if not canonical:
            return pl.DataFrame(
                schema={
                    "timestamp": pl.Utf8,
                    "subject": pl.Utf8,
                    "action": pl.Utf8,
                    "verdict": pl.Utf8,
                    "reason": pl.Utf8,
                    "hash": pl.Utf8,
                    "prev_hash": pl.Utf8,
                }
            )
        # Polars infers types; coerce timestamps / subjects to string for display.
        return pl.DataFrame(canonical).with_columns(
            [
                pl.col("timestamp").cast(pl.Utf8, strict=False),
                pl.col("subject").cast(pl.Utf8, strict=False),
                pl.col("action").cast(pl.Utf8, strict=False),
                pl.col("verdict").cast(pl.Utf8, strict=False),
                pl.col("reason").cast(pl.Utf8, strict=False),
                pl.col("hash").cast(pl.Utf8, strict=False),
                pl.col("prev_hash").cast(pl.Utf8, strict=False),
            ]
        )

    # ── Chain integrity ────────────────────────────────────────────────

    def verify_chain(
        self, audit_log: pl.DataFrame | Sequence[dict[str, Any]]
    ) -> pl.DataFrame:
        """Check hash-chain integrity.

        Contract: each row's ``hash`` must equal ``sha256(prev_hash +
        canonical_json(row_data))``. Rows lacking ``hash`` / ``prev_hash``
        are skipped and flagged as ``unchecked``.

        Returns a Polars DataFrame with one row per audit entry and a
        per-row ``ok`` / ``broken`` / ``unchecked`` verdict. The caller
        can ``filter(pl.col("integrity") == "broken")`` to surface tampering.
        """
        if isinstance(audit_log, pl.DataFrame):
            rows = audit_log.to_dicts()
        else:
            rows = list(audit_log)

        report_rows: list[dict[str, Any]] = []
        n_broken = 0
        n_ok = 0
        n_unchecked = 0
        expected_prev: str | None = None

        for idx, r in enumerate(rows):
            h = r.get("hash")
            prev = r.get("prev_hash")
            if not h:
                integrity = "unchecked"
                n_unchecked += 1
            else:
                payload = {k: v for k, v in r.items() if k not in {"hash"}}
                # Canonical JSON — sort keys, no whitespace, default=str so
                # datetimes round-trip.
                serialised = json.dumps(payload, sort_keys=True, default=str).encode(
                    "utf-8"
                )
                recomputed = hashlib.sha256(
                    ((prev or "").encode("utf-8") + serialised)
                ).hexdigest()
                chain_ok = recomputed == h
                link_ok = (expected_prev is None) or (prev == expected_prev)
                if chain_ok and link_ok:
                    integrity = "ok"
                    n_ok += 1
                else:
                    integrity = "broken"
                    n_broken += 1
                expected_prev = h

            report_rows.append(
                {
                    "idx": idx,
                    "timestamp": r.get("timestamp"),
                    "hash": h,
                    "prev_hash": prev,
                    "integrity": integrity,
                }
            )

        logger.info(
            "governance.verify_chain",
            extra={
                "run_id": self._run_id,
                "rows": len(rows),
                "ok": n_ok,
                "broken": n_broken,
                "unchecked": n_unchecked,
                "source": "local_metric",
                "mode": "real",
            },
        )
        if n_broken:
            logger.warning(
                "governance.chain_broken",
                extra={"run_id": self._run_id, "broken": n_broken, "total": len(rows)},
            )

        if not report_rows:
            return pl.DataFrame(
                schema={
                    "idx": pl.Int64,
                    "timestamp": pl.Utf8,
                    "hash": pl.Utf8,
                    "prev_hash": pl.Utf8,
                    "integrity": pl.Utf8,
                }
            )
        return pl.DataFrame(report_rows).with_columns(
            [
                pl.col("timestamp").cast(pl.Utf8, strict=False),
                pl.col("hash").cast(pl.Utf8, strict=False),
                pl.col("prev_hash").cast(pl.Utf8, strict=False),
            ]
        )

    # ── Budget consumption ─────────────────────────────────────────────

    def budget_consumption(self) -> pl.DataFrame:
        """Aggregate envelope usage per dimension into a Polars DataFrame.

        Columns:

            * ``dimension`` — one of ``financial``, ``temporal``,
              ``data_access``, ``communication``, ``operational``.
            * ``limit`` — envelope cap as declared by the engine.
            * ``consumed`` — amount consumed so far.
            * ``remaining`` — ``limit - consumed`` (clipped at 0).
            * ``pct_used`` — ``consumed / limit`` in ``[0, 1]`` (``nan`` if
              the limit is unset).
        """
        engine = self._require_engine("budget_consumption")

        # Canonical shape 1: engine exposes ``.budgets`` dict
        # {"financial": {"limit": X, "consumed": Y}, ...}.
        budgets: dict[str, dict[str, float]] = {}
        if hasattr(engine, "budgets"):
            raw = engine.budgets() if callable(engine.budgets) else engine.budgets
            if isinstance(raw, dict):
                budgets = {
                    str(k): dict(v) for k, v in raw.items() if isinstance(v, dict)
                }

        # Canonical shape 2: derive from audit log — count rows by dimension
        # with a "consumed" / "amount" field.
        if not budgets:
            rows = self._extract_audit_rows(engine)
            tallies: dict[str, float] = {d: 0.0 for d in _ENVELOPE_DIMENSIONS}
            for r in rows:
                dim = r.get("dimension") or r.get("envelope_dimension")
                amount = r.get("amount") or r.get("consumed") or 0.0
                if dim in tallies:
                    try:
                        tallies[dim] += float(amount)
                    except (TypeError, ValueError):
                        continue
            budgets = {
                d: {"limit": float("nan"), "consumed": tallies[d]}
                for d in _ENVELOPE_DIMENSIONS
            }

        rows_out: list[dict[str, Any]] = []
        for dim in _ENVELOPE_DIMENSIONS:
            b = budgets.get(dim, {})
            limit = float(b.get("limit", float("nan")))
            consumed = float(b.get("consumed", 0.0))
            remaining = max(0.0, limit - consumed) if limit == limit else float("nan")
            pct = (consumed / limit) if (limit == limit and limit > 0) else float("nan")
            rows_out.append(
                {
                    "dimension": dim,
                    "limit": limit,
                    "consumed": consumed,
                    "remaining": remaining,
                    "pct_used": pct,
                }
            )

        logger.info(
            "governance.budget_consumption",
            extra={
                "run_id": self._run_id,
                "dimensions": len(rows_out),
                "source": "governance_engine",
                "mode": "real",
            },
        )
        return pl.DataFrame(rows_out)

    # ── Negative drills ────────────────────────────────────────────────

    def negative_drills(self, scenarios: Sequence[dict[str, Any]]) -> pl.DataFrame:
        """Run red-team scenarios through ``GovernanceEngine.verify_action``.

        Each scenario dict is passed as ``**kwargs`` to the engine's
        ``verify_action`` (or ``verify``) method. The returned object's
        ``verdict`` / ``decision`` field is recorded.

        Args:
            scenarios: List of kwarg dicts describing the action to test.
                e.g. ``[{"subject": "agent-1", "action": "transfer",
                "amount": 1_000_000, "dimension": "financial"}]``.

        Returns:
            A Polars DataFrame with one row per scenario: ``scenario``,
            ``verdict``, ``reason``, ``mode``.
        """
        engine = self._require_engine("negative_drills")
        verify = getattr(engine, "verify_action", None) or getattr(
            engine, "verify", None
        )
        if verify is None or not callable(verify):
            raise AttributeError(
                "Governance engine does not expose verify_action(). "
                "Pass a PACT GovernanceEngine or a GovernedSupervisor.audit "
                "that supports verification."
            )

        results: list[_DrillResult] = []
        for i, sc in enumerate(scenarios):
            label = sc.get("label") or sc.get("action") or f"drill_{i}"
            try:
                outcome = verify(**sc) if isinstance(sc, dict) else verify(sc)
            except Exception as exc:
                logger.exception(
                    "governance.drill.error",
                    extra={"scenario": label, "error": str(exc)},
                )
                results.append(
                    _DrillResult(
                        scenario=label,
                        verdict="error",
                        reason=str(exc),
                        mode="real",
                    )
                )
                continue

            verdict = _extract_field(
                outcome, ("verdict", "decision"), default="unknown"
            )
            reason = _extract_field(outcome, ("reason", "rationale"), default="")
            results.append(
                _DrillResult(
                    scenario=label,
                    verdict=str(verdict).lower(),
                    reason=str(reason),
                    mode="real",
                )
            )

        self._drill_results.extend(results)
        logger.info(
            "governance.negative_drills",
            extra={
                "run_id": self._run_id,
                "scenarios": len(scenarios),
                "source": "governance_engine",
                "mode": "real",
            },
        )
        return pl.DataFrame(
            [
                {
                    "scenario": r.scenario,
                    "verdict": r.verdict,
                    "reason": r.reason,
                    "mode": r.mode,
                }
                for r in results
            ]
        )

    # ── Plot ───────────────────────────────────────────────────────────

    def plot_governance_dashboard(self) -> go.Figure:
        """2x2 dashboard: verdict-over-time, budget bars, drill heatmap, chain timeline."""
        from plotly.subplots import make_subplots

        if self._engine is None:
            return empty_figure(
                "Governance Lens Dashboard (Flight Recorder)",
                note="no governance engine",
            )

        fig = make_subplots(
            rows=2,
            cols=2,
            subplot_titles=(
                "Verdicts over time (stacked)",
                "Envelope budget consumption",
                "Negative-drill verdict mix",
                "Chain integrity timeline",
            ),
        )

        # (1,1) Verdict-over-time: stacked by verdict.
        try:
            snap = self.audit_snapshot(last_n=500)
        except Exception as exc:  # engine refused
            logger.warning(
                "governance.plot.snapshot_skipped", extra={"error": str(exc)}
            )
            snap = pl.DataFrame()
        if snap.height:
            verdicts_seen = snap["verdict"].unique().to_list()
            for verdict in verdicts_seen:
                sub = snap.filter(pl.col("verdict") == verdict)
                fig.add_trace(
                    go.Scatter(
                        x=(
                            sub["timestamp"].to_list()
                            if "timestamp" in sub.columns
                            else list(range(sub.height))
                        ),
                        y=[1] * sub.height,
                        mode="markers",
                        marker=dict(
                            color=_VERDICT_COLORS.get(verdict, MUTED),
                            size=8,
                        ),
                        name=verdict,
                        stackgroup="verdicts",
                    ),
                    row=1,
                    col=1,
                )

        # (1,2) Budget consumption bars.
        try:
            budget_df = self.budget_consumption()
        except Exception as exc:
            logger.warning("governance.plot.budget_skipped", extra={"error": str(exc)})
            budget_df = pl.DataFrame()
        if budget_df.height:
            fig.add_trace(
                go.Bar(
                    x=budget_df["dimension"].to_list(),
                    y=budget_df["consumed"].to_list(),
                    marker_color=PRIMARY,
                    name="consumed",
                ),
                row=1,
                col=2,
            )

        # (2,1) Negative drill verdict mix.
        if self._drill_results:
            counts: dict[str, int] = {}
            for r in self._drill_results:
                counts[r.verdict] = counts.get(r.verdict, 0) + 1
            fig.add_trace(
                go.Bar(
                    x=list(counts.keys()),
                    y=list(counts.values()),
                    marker_color=[_VERDICT_COLORS.get(v, MUTED) for v in counts],
                    name="drills",
                ),
                row=2,
                col=1,
            )

        # (2,2) Chain integrity timeline.
        if snap.height:
            chain_df = self.verify_chain(snap)
            color_map = {
                "ok": PRIMARY,
                "broken": WARN,
                "unchecked": MUTED,
            }
            fig.add_trace(
                go.Scatter(
                    x=chain_df["idx"].to_list(),
                    y=[1] * chain_df.height,
                    mode="markers",
                    marker=dict(
                        color=[
                            color_map.get(v, MUTED)
                            for v in chain_df["integrity"].to_list()
                        ],
                        size=10,
                        symbol="square",
                    ),
                    name="chain",
                ),
                row=2,
                col=2,
            )

        fig.update_layout(
            title="Governance Lens Dashboard (Flight Recorder)",
            template=TEMPLATE,
            showlegend=True,
            height=640,
        )
        return fig

    # ── Report ─────────────────────────────────────────────────────────

    def report(self) -> str:
        """Plain-text Prescription Pad for the governance lens."""
        if self._engine is None:
            return (
                "governance-lens: no engine configured — pass "
                "governance=GovernanceEngine(...) to read envelope state."
            )

        out: list[str] = []
        try:
            snap = self.audit_snapshot(last_n=200)
        except Exception as exc:
            return f"governance-lens: snapshot error — {exc}"

        if snap.height:
            verdict_counts = (
                snap.group_by("verdict")
                .agg(pl.len().alias("n"))
                .sort("n", descending=True)
            )
            parts = [
                f"{row['verdict']}={row['n']}"
                for row in verdict_counts.iter_rows(named=True)
            ]
            out.append(f"audit: {snap.height} entries — {', '.join(parts)}")

            chain_df = self.verify_chain(snap)
            n_broken = chain_df.filter(pl.col("integrity") == "broken").height
            n_unchecked = chain_df.filter(pl.col("integrity") == "unchecked").height
            if n_broken:
                out.append(
                    f"  -> CHAIN BROKEN at {n_broken} row(s) — possible tampering"
                )
            elif n_unchecked == chain_df.height:
                out.append("  -> chain unchecked (no hash/prev_hash columns present)")
            else:
                out.append(
                    f"  -> chain ok ({chain_df.height - n_broken - n_unchecked} linked)"
                )

        try:
            budget_df = self.budget_consumption()
        except Exception as exc:
            budget_df = None
            out.append(f"budget: unavailable — {exc}")
        if budget_df is not None and budget_df.height:
            for row in budget_df.iter_rows(named=True):
                pct = row["pct_used"]
                if pct == pct and pct >= 0.8:  # not NaN and >= 80%
                    out.append(
                        f"  -> envelope[{row['dimension']}] at {pct:.0%} — "
                        "approaching cap"
                    )

        if self._drill_results:
            blocked = sum(
                1 for r in self._drill_results if r.verdict in {"block", "escalate"}
            )
            total = len(self._drill_results)
            out.append(
                f"drills: {total} scenarios — {blocked}/{total} blocked/escalated"
            )
            if blocked < total:
                out.append(
                    "  -> some drills passed — inspect scenarios that were allowed"
                )

        if not out:
            return "governance-lens: no readings recorded yet."
        return "governance-lens:\n  " + "\n  ".join(out)


# ════════════════════════════════════════════════════════════════════════
# Helpers
# ════════════════════════════════════════════════════════════════════════


def _extract_field(obj: Any, keys: Sequence[str], *, default: Any = None) -> Any:
    """Pull the first matching key from a dict / dataclass / attr object."""
    if obj is None:
        return default
    if isinstance(obj, dict):
        for k in keys:
            if k in obj:
                return obj[k]
        return default
    for k in keys:
        if hasattr(obj, k):
            return getattr(obj, k)
    return default


# ── shared/mlfp06/diagnostics/agent.py ──
"""Lens 4 — Agent Trace Diagnostics (the MRI).

Question answered: *What did the agent do, and where did it spend its budget?*

Consumes the event stream from a Kaizen :class:`~kaizen_agents.Delegate`
(the TAOD loop — Thought/Action/Observation/Decision) and produces:

    * a TAOD-structured :class:`~shared.mlfp06.diagnostics.AgentTrace`
    * tool-usage, cost, and latency breakdowns as Polars DataFrames
    * stuck-loop and oscillation detection
    * a timeline Plotly figure suitable for both Colab and VS Code
    * optional Langfuse export when ``LANGFUSE_HOST`` is set

Raw ``openai.*`` calls are BLOCKED (``rules/framework-first.md``). All
agent execution goes through ``Delegate.run()`` — we only observe its
event stream.

Quick start::


    agent_diag = AgentDiagnostics()
    delegate = make_delegate(tools=[...])  # Ollama-backed; no API keys

    trace = await agent_diag.capture_run(delegate, "Research and summarise ...")
    agent_diag.plot_trace(trace.run_id).show()
    print(agent_diag.report(trace.run_id))
"""

import inspect
import logging
import os
import time
import uuid
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable, Sequence

import plotly.graph_objects as go
import polars as pl


logger = logging.getLogger(__name__)

__all__ = ["AgentDiagnostics", "AgentTrace"]


# Minimum number of repeated (tool, args-hash) pairs in a window that we
# consider a "loop" for stuck-pattern detection.
_LOOP_MIN_REPEATS = 3


@dataclass
class _LoopFinding:
    run_id: str
    tool: str
    signature: str
    repeats: int
    first_idx: int
    last_idx: int


class AgentDiagnostics:
    """Agent-lens diagnostics — TAOD trace capture, timeline, stuck detection.

    Args:
        traces_dir: Directory where JSONL traces are persisted. When
            ``None`` traces live in-memory only (Colab default).
        langfuse_host: If set (or ``LANGFUSE_HOST`` env var), captured
            traces are exported after each run. When the Langfuse SDK is
            missing, export is skipped with a WARN log.
        langfuse_public_key / langfuse_secret_key: Auth for Langfuse; fall
            back to ``LANGFUSE_PUBLIC_KEY`` / ``LANGFUSE_SECRET_KEY``.
    """

    def __init__(
        self,
        *,
        traces_dir: Path | str | None = None,
        langfuse_host: str | None = None,
        langfuse_public_key: str | None = None,
        langfuse_secret_key: str | None = None,
    ) -> None:
        self._traces: dict[str, AgentTrace] = {}
        self._loops: list[_LoopFinding] = []
        self._traces_dir: Path | None = (
            Path(traces_dir) if traces_dir is not None else None
        )
        if self._traces_dir is not None:
            self._traces_dir.mkdir(parents=True, exist_ok=True)

        self._langfuse_host = langfuse_host or os.environ.get("LANGFUSE_HOST")
        self._langfuse_public = langfuse_public_key or os.environ.get(
            "LANGFUSE_PUBLIC_KEY"
        )
        self._langfuse_secret = langfuse_secret_key or os.environ.get(
            "LANGFUSE_SECRET_KEY"
        )
        self._langfuse_client: Any = None

        logger.info(
            "agent_diagnostics.init",
            extra={
                "traces_dir": str(self._traces_dir) if self._traces_dir else None,
                "langfuse_enabled": bool(self._langfuse_host),
            },
        )

    def __enter__(self) -> "AgentDiagnostics":
        return self

    def __exit__(self, *exc: Any) -> None:
        self.close()

    def close(self) -> None:
        for t in self.values():
            t.close()
        if self._langfuse_client is not None:
            try:
                # Langfuse uses ``flush`` to drain queued spans.
                self._langfuse_client.flush()
            except Exception:
                # Cleanup path — zero-tolerance Rule 3 carve-out.
                pass

    # ── Run capture ────────────────────────────────────────────────────

    async def capture_run(
        self,
        delegate: Any,
        prompt: str,
        *,
        run_id: str | None = None,
        tags: Sequence[str] = (),
    ) -> AgentTrace:
        """Execute ``delegate.run(prompt)`` and record its event stream.

        The delegate's ``run()`` is expected to be an async iterator of
        Kaizen ``StreamEvent`` objects (per ``kaizen_agents`` 0.9 API).
        Each event is routed through
        :func:`~shared.mlfp06.diagnostics.kaizen_events_to_trace`
        to produce a TAOD :class:`AgentTrace`.

        Args:
            delegate: A constructed ``kaizen_agents.Delegate`` (or a mock
                exposing the same ``run(prompt)`` signature).
            prompt: The user prompt to feed the agent.
            run_id: Correlation ID; auto-generated when ``None``.
            tags: Optional tags forwarded to Langfuse on export.

        Returns:
            The :class:`AgentTrace` recorded for the run.
        """
        run_id = run_id or f"agent-{uuid.uuid4().hex[:12]}"
        t0 = time.monotonic()
        logger.info(
            "agent.capture_run.start",
            extra={"run_id": run_id, "prompt_preview": prompt[:80], "mode": "real"},
        )

        path: Path | None = None
        if self._traces_dir is not None:
            path = self._traces_dir / f"{run_id}.jsonl"

        events = await _collect_delegate_events(delegate, prompt)
        trace = kaizen_events_to_trace(events, run_id=run_id, path=path)
        self._traces[run_id] = trace

        # Post-capture analysis.
        for finding in _detect_tool_loops(trace):
            self._loops.append(finding)

        latency = (time.monotonic() - t0) * 1000.0
        logger.info(
            "agent.capture_run.ok",
            extra={
                "run_id": run_id,
                "n_events": len(trace),
                "cost_usd": trace.total_cost_usd(),
                "latency_ms": latency,
                "mode": "real",
            },
        )

        if self._langfuse_host:
            self._export_to_langfuse(trace, prompt=prompt, tags=tuple(tags))
        return trace

    def register_trace(self, trace: AgentTrace) -> AgentTrace:
        """Register an externally-produced trace for analysis.

        Useful when the trace was captured out-of-band (Langfuse SDK,
        OTEL exporter) and the user wants the lens's analysis methods to
        operate on it.
        """
        self._traces[trace.run_id] = trace
        for finding in _detect_tool_loops(trace):
            self._loops.append(finding)
        return trace

    # ── Retrieval / queries ────────────────────────────────────────────

    def get(self, run_id: str) -> AgentTrace:
        if run_id not in self._traces:
            raise KeyError(f"No trace registered for run_id={run_id!r}")
        return self._traces[run_id]

    def run_ids(self) -> list[str]:
        return list(self.keys())

    def tool_usage(self, run_id: str) -> pl.DataFrame:
        """Tool-call counts with mean latency + total cost per tool."""
        trace = self.get(run_id)
        rows: dict[str, dict[str, float]] = {}
        for ev in trace.events:
            if ev.kind not in {"tool_start", "tool_end", "error"}:
                continue
            name = ev.tool or "<unknown>"
            bucket = rows.setdefault(
                name,
                {
                    "calls": 0.0,
                    "errors": 0.0,
                    "latency_ms_sum": 0.0,
                    "cost_usd_sum": 0.0,
                },
            )
            if ev.kind == "tool_start":
                bucket["calls"] += 1
            else:
                bucket["latency_ms_sum"] += ev.latency_ms
                bucket["cost_usd_sum"] += ev.cost_usd
                if ev.kind == "error" or ev.error:
                    bucket["errors"] += 1
        if not rows:
            return pl.DataFrame(
                schema={
                    "tool": pl.Utf8,
                    "calls": pl.Int64,
                    "errors": pl.Int64,
                    "mean_latency_ms": pl.Float64,
                    "total_cost_usd": pl.Float64,
                }
            )
        out = []
        for name, b in rows.items():
            calls = int(b["calls"])
            out.append(
                {
                    "tool": name,
                    "calls": calls,
                    "errors": int(b["errors"]),
                    "mean_latency_ms": (b["latency_ms_sum"] / calls if calls else 0.0),
                    "total_cost_usd": b["cost_usd_sum"],
                }
            )
        return pl.DataFrame(out).sort("calls", descending=True)

    def cost_breakdown(self, run_id: str) -> pl.DataFrame:
        """Per-event cost + cumulative cost for the run."""
        trace = self.get(run_id)
        running = 0.0
        rows = []
        for idx, ev in enumerate(trace.events):
            running += ev.cost_usd
            rows.append(
                {
                    "idx": idx,
                    "kind": ev.kind,
                    "tool": ev.tool,
                    "cost_usd": ev.cost_usd,
                    "cum_cost_usd": running,
                    "latency_ms": ev.latency_ms,
                }
            )
        if not rows:
            return pl.DataFrame(
                schema={
                    "idx": pl.Int64,
                    "kind": pl.Utf8,
                    "tool": pl.Utf8,
                    "cost_usd": pl.Float64,
                    "cum_cost_usd": pl.Float64,
                    "latency_ms": pl.Float64,
                }
            )
        return pl.DataFrame(rows)

    def detect_loops(self, run_id: str | None = None) -> pl.DataFrame:
        """Find repeated (tool, args) action sequences.

        A "loop" is a ``(tool, args-signature)`` pair that appears at
        least ``_LOOP_MIN_REPEATS`` times consecutively in the trace.
        """
        findings = (
            [f for f in self._loops if f.run_id == run_id]
            if run_id is not None
            else list(self._loops)
        )
        if not findings:
            return pl.DataFrame(
                schema={
                    "run_id": pl.Utf8,
                    "tool": pl.Utf8,
                    "signature": pl.Utf8,
                    "repeats": pl.Int64,
                    "first_idx": pl.Int64,
                    "last_idx": pl.Int64,
                }
            )
        return pl.DataFrame(
            [
                {
                    "run_id": f.run_id,
                    "tool": f.tool,
                    "signature": f.signature[:80],
                    "repeats": f.repeats,
                    "first_idx": f.first_idx,
                    "last_idx": f.last_idx,
                }
                for f in findings
            ]
        )

    # ── Plots ──────────────────────────────────────────────────────────

    def plot_trace(self, run_id: str) -> go.Figure:
        """2-row timeline: (tool calls on a time axis) + (cumulative cost)."""
        trace = self.get(run_id)
        if not len(trace):
            return empty_figure(f"Agent Trace {run_id}", note="no events")
        from plotly.subplots import make_subplots

        fig = make_subplots(
            rows=2,
            cols=1,
            shared_xaxes=True,
            subplot_titles=("Tool calls (timeline)", "Cumulative cost (USD)"),
            row_heights=[0.6, 0.4],
            vertical_spacing=0.08,
        )

        # Row 1 — tool call markers with latency as bar height.
        tool_rows = self.cost_breakdown(run_id).filter(
            pl.col("kind").is_in(["tool_start", "tool_end", "error"])
        )
        if tool_rows.height:
            unique_tools = sorted(
                {t for t in tool_rows["tool"].to_list() if t is not None}
            )
            for i, t in enumerate(unique_tools):
                sub = tool_rows.filter(pl.col("tool") == t)
                fig.add_trace(
                    go.Scatter(
                        x=sub["idx"].to_list(),
                        y=[i] * sub.height,
                        mode="markers+lines",
                        name=t,
                        marker=dict(
                            size=10, color=color_for(i), symbol="square"
                        ),
                        hovertext=sub["latency_ms"]
                        .map_elements(
                            lambda v: f"latency={v:.0f}ms", return_dtype=pl.Utf8
                        )
                        .to_list(),
                    ),
                    row=1,
                    col=1,
                )
            fig.update_yaxes(
                tickmode="array",
                tickvals=list(range(len(unique_tools))),
                ticktext=unique_tools,
                row=1,
                col=1,
            )

        # Row 2 — cumulative cost curve.
        cost_df = self.cost_breakdown(run_id)
        if cost_df.height:
            fig.add_trace(
                go.Scatter(
                    x=cost_df["idx"].to_list(),
                    y=cost_df["cum_cost_usd"].to_list(),
                    mode="lines",
                    line=dict(color=WARN, width=2),
                    name="cum cost",
                ),
                row=2,
                col=1,
            )

        # Loop markers (vertical bands) when stuck patterns are present.
        for finding in self._loops:
            if finding.run_id != run_id:
                continue
            fig.add_vrect(
                x0=finding.first_idx,
                x1=finding.last_idx,
                fillcolor=ACCENT,
                opacity=0.15,
                line_width=0,
                annotation_text=f"loop x{finding.repeats}",
                annotation_position="top left",
                row=1,
                col=1,
            )

        fig.update_layout(
            title=f"Agent MRI — {run_id}",
            template=TEMPLATE,
            height=620,
            showlegend=True,
        )
        return fig

    def plot_cost_across_runs(self) -> go.Figure:
        """Bar chart of total cost per captured run."""
        if not self._traces:
            return empty_figure("Agent cost across runs")
        rows = [
            {
                "run_id": rid,
                "cost_usd": trace.total_cost_usd(),
                "latency_ms": trace.total_latency_ms(),
                "events": len(trace),
            }
            for rid, trace in self.items()
        ]
        df = pl.DataFrame(rows).sort("cost_usd", descending=True)
        fig = go.Figure(
            go.Bar(
                x=df["run_id"].to_list(),
                y=df["cost_usd"].to_list(),
                marker_color=PRIMARY,
            )
        )
        style(fig, "Agent cost per run", x="run_id", y="cost (USD)")
        return fig

    # ── Report ─────────────────────────────────────────────────────────

    def report(self, run_id: str | None = None) -> str:
        """Plain-text Prescription Pad — one run or all runs."""
        target_ids = [run_id] if run_id else list(self.keys())
        if not target_ids:
            return "agent-lens: no runs captured yet."

        out: list[str] = []
        for rid in target_ids:
            if rid not in self._traces:
                continue
            trace = self._traces[rid]
            tool_df = self.tool_usage(rid)
            cost = trace.total_cost_usd()
            latency = trace.total_latency_ms()
            out.append(
                f"run={rid}: {len(trace)} events, cost=${cost:.4f}, "
                f"latency={latency:.0f}ms, tools_used={tool_df.height}"
            )
            if tool_df.height:
                top = tool_df.row(0, named=True)
                out.append(
                    f"  top_tool={top['tool']} calls={top['calls']} "
                    f"mean_latency={top['mean_latency_ms']:.0f}ms "
                    f"errors={top['errors']}"
                )
            errors = sum(1 for ev in trace.events if ev.kind == "error" or ev.error)
            if errors:
                out.append(f"  -> {errors} error event(s) — inspect trace")
            loops_here = [f for f in self._loops if f.run_id == rid]
            if loops_here:
                out.append(
                    f"  -> stuck loop detected: {loops_here[0].tool} x"
                    f"{loops_here[0].repeats} "
                    f"(idx {loops_here[0].first_idx}..{loops_here[0].last_idx})"
                )
        return "agent-lens:\n  " + "\n  ".join(out)

    # ── Langfuse export ────────────────────────────────────────────────

    def _export_to_langfuse(
        self, trace: AgentTrace, *, prompt: str, tags: tuple[str, ...]
    ) -> None:
        if not self._langfuse_host:
            return
        client = self._ensure_langfuse()
        if client is None:
            return
        try:
            lf_trace = client.trace(
                name="mlfp06.agent.run",
                id=trace.run_id,
                input=prompt,
                metadata={"n_events": len(trace)},
                tags=list(tags),
            )
            for ev in trace.events:
                if ev.kind in {"tool_start", "tool_end", "error"} and ev.tool:
                    lf_trace.span(
                        name=f"tool:{ev.tool}",
                        input=ev.args or {},
                        output=ev.result,
                        metadata={
                            "latency_ms": ev.latency_ms,
                            "cost_usd": ev.cost_usd,
                            "error": ev.error,
                        },
                    )
                elif ev.kind == "complete":
                    lf_trace.generation(
                        name="agent.complete",
                        output=ev.content,
                        usage={
                            "input": ev.tokens_in or 0,
                            "output": ev.tokens_out or 0,
                        },
                    )
            logger.info(
                "agent.langfuse.export.ok",
                extra={"run_id": trace.run_id, "host": self._langfuse_host},
            )
        except Exception as exc:  # pragma: no cover — network-dependent
            logger.warning(
                "agent.langfuse.export.error",
                extra={"run_id": trace.run_id, "error": str(exc)},
            )

    def _ensure_langfuse(self) -> Any:
        if self._langfuse_client is not None:
            return self._langfuse_client
        try:
            # langfuse 2.x exposes the client as ``Langfuse``.
            from langfuse import Langfuse  # type: ignore[import-not-found]
        except ImportError:
            logger.warning(
                "agent.langfuse.unavailable",
                extra={
                    "reason": "langfuse package not installed",
                    "host": self._langfuse_host,
                },
            )
            return None
        try:
            self._langfuse_client = Langfuse(
                host=self._langfuse_host,
                public_key=self._langfuse_public,
                secret_key=self._langfuse_secret,
            )
        except Exception as exc:  # pragma: no cover — credential-dependent
            logger.warning(
                "agent.langfuse.construct_error",
                extra={"error": str(exc), "host": self._langfuse_host},
            )
            return None
        return self._langfuse_client


# ════════════════════════════════════════════════════════════════════════
# Helpers
# ════════════════════════════════════════════════════════════════════════


async def _collect_delegate_events(delegate: Any, prompt: str) -> list[Any]:
    """Drain ``delegate.run(prompt)`` into a list of StreamEvents.

    The Kaizen 0.9 API returns an async iterator. We support three shapes:

        1. ``async for event in delegate.run(prompt): ...`` (primary)
        2. ``for event in delegate.run(prompt): ...`` (sync iterator — used
           by tests that want zero async boilerplate)
        3. a pre-collected list (returned directly by mocks)
    """
    call = delegate.run(prompt)
    # Case 3 — already a list / tuple.
    if isinstance(call, (list, tuple)):
        return list(call)
    # Case 1 — async iterator.
    if hasattr(call, "__aiter__"):
        events: list[Any] = []
        async for event in call:
            events.append(event)
        return events
    # Coroutine returning iterable.
    if inspect.iscoroutine(call):
        result = await call
        if isinstance(result, (list, tuple)):
            return list(result)
        if hasattr(result, "__aiter__"):
            events = []
            async for event in result:
                events.append(event)
            return events
        if hasattr(result, "__iter__"):
            return list(result)
        return [result]
    # Case 2 — sync iterator.
    if hasattr(call, "__iter__"):
        return list(call)
    return [call]


def _arg_signature(args: Any) -> str:
    """Stable short string for ``(tool, args)`` equality checks."""
    if args is None:
        return "()"
    try:
        import json

        return json.dumps(args, sort_keys=True, default=str)[:200]
    except Exception:
        return repr(args)[:200]


def _detect_tool_loops(trace: AgentTrace) -> list[_LoopFinding]:
    """Identify consecutive runs of the same (tool, args) signature."""
    findings: list[_LoopFinding] = []
    run_id = trace.run_id
    prev_sig: str | None = None
    prev_tool: str | None = None
    count = 0
    first_idx = 0
    last_idx = 0

    for idx, ev in enumerate(trace.events):
        if ev.kind != "tool_start":
            continue
        sig = f"{ev.tool}|{_arg_signature(ev.args)}"
        if sig == prev_sig:
            count += 1
            last_idx = idx
        else:
            if prev_sig is not None and count >= _LOOP_MIN_REPEATS:
                findings.append(
                    _LoopFinding(
                        run_id=run_id,
                        tool=prev_tool or "<unknown>",
                        signature=prev_sig,
                        repeats=count,
                        first_idx=first_idx,
                        last_idx=last_idx,
                    )
                )
            prev_sig = sig
            prev_tool = ev.tool
            count = 1
            first_idx = idx
            last_idx = idx
    if prev_sig is not None and count >= _LOOP_MIN_REPEATS:
        findings.append(
            _LoopFinding(
                run_id=run_id,
                tool=prev_tool or "<unknown>",
                signature=prev_sig,
                repeats=count,
                first_idx=first_idx,
                last_idx=last_idx,
            )
        )
    return findings


# ── shared/mlfp06/diagnostics/observatory.py ──
"""LLM Observatory — central facade composing all six clinical lenses.

Mirrors ``kailash_ml.diagnostics.DLDiagnostics`` (the M5 Doctor's Bag)
for the M6 problem domain. The facade gives M6 exercises one import and
one constructor call that wires up every lens:

    * :class:`~shared.mlfp06.diagnostics.output.LLMDiagnostics` — Output
    * :class:`~shared.mlfp06.diagnostics.interpretability.InterpretabilityDiagnostics`
      — Attention
    * :class:`~shared.mlfp06.diagnostics.retrieval.RAGDiagnostics` — Retrieval
    * :class:`~shared.mlfp06.diagnostics.agent.AgentDiagnostics` — Agent trace
    * :class:`~shared.mlfp06.diagnostics.alignment.AlignmentDiagnostics`
      — Alignment
    * :class:`~shared.mlfp06.diagnostics.governance.GovernanceDiagnostics`
      — Governance

Typical use::


    with LLMObservatory(delegate=my_delegate,
                        governance=supervisor.audit) as obs:
        obs.output.evaluate(prompts, responses)
        obs.retrieval.evaluate(queries, contexts, answers)
        obs.agent.capture_run(stream_events, run_id="demo")
        print(obs.report())
        obs.plot_dashboard().show()

All lenses degrade gracefully when their inputs are absent — the facade
never fabricates fake readings. When a method that requires a live
dependency (e.g. judge delegate, governance engine) is called without
one, the underlying lens raises a loud error per
``rules/observability.md`` §3.1 (no silent fake verdicts).
"""

import logging
from typing import Any

import plotly.graph_objects as go


logger = logging.getLogger(__name__)

__all__ = ["LLMObservatory"]


# Severity thresholds — centralised here so every lens scores on the
# same axis when the facade composes a dashboard. Per
# ``rules/agent-reasoning.md``: these are NOT agent decisions. They are
# deterministic quality thresholds over already-computed metrics, which
# is explicit exception 4 (safety guards) in the permitted-conditional
# list.
_SEVERITY_HEALTHY = "HEALTHY"
_SEVERITY_WARNING = "WARNING"
_SEVERITY_CRITICAL = "CRITICAL"
_SEVERITY_UNKNOWN = "UNKNOWN"


class LLMObservatory:
    """Compose all six lenses — central entry point for M6 exercises.

    Args:
        delegate: Kaizen ``Delegate`` under observation. Re-used as the
            judge for the Output and Retrieval lenses. Optional — lenses
            that need a delegate raise loudly when called without one.
        judge_model: Explicit judge model. ``None`` resolves via
            ``OPENAI_JUDGE_MODEL`` → ``DEFAULT_LLM_MODEL`` →
            ``OPENAI_PROD_MODEL`` (see ``rules/env-models.md``).
        governance: ``GovernedSupervisor.audit`` or a PACT
            ``GovernanceEngine`` — whatever the Governance lens should
            observe.
        run_id: Correlation ID propagated to every lens for
            observability. Auto-generated if ``None``.
        max_judge_calls: Budget cap shared by Output and Retrieval
            judges. Defaults to 50 per ``JudgeCallable``.
        attention_model: Open-weight model name for the attention lens
            (defaults to the lens's own default). API-only models short-
            circuit with a ``not_applicable`` verdict.
    """

    def __init__(
        self,
        *,
        delegate: Any = None,
        judge_model: str | None = None,
        governance: Any = None,
        run_id: str | None = None,
        max_judge_calls: int = 50,
        attention_model: str | None = None,
    ) -> None:
        self._run_id = run_id

        self.output = LLMDiagnostics(
            delegate=delegate,
            judge_model=judge_model,
            max_judge_calls=max_judge_calls,
        )

        # InterpretabilityDiagnostics accepts model_name + run_id_prefix;
        # route run_id through the prefix for correlation.
        attention_kwargs: dict[str, Any] = {}
        if attention_model is not None:
            attention_kwargs["model_name"] = attention_model
        if run_id is not None:
            attention_kwargs["run_id_prefix"] = run_id
        self.attention = InterpretabilityDiagnostics(**attention_kwargs)

        self.retrieval = RAGDiagnostics(
            delegate=delegate,
            judge_model=judge_model,
            max_judge_calls=max_judge_calls,
        )

        self.agent = AgentDiagnostics()

        # AlignmentDiagnostics uses label= not run_id=.
        self.alignment = AlignmentDiagnostics(
            label=run_id if run_id is not None else "run"
        )

        self.governance = GovernanceDiagnostics(
            governance=governance,
            run_id=run_id,
        )

        self._delegate = delegate
        self._governance_engine = governance
        logger.info(
            "observatory.init",
            extra={
                "run_id": run_id,
                "has_delegate": delegate is not None,
                "has_governance": governance is not None,
                "judge_model": judge_model,
                "max_judge_calls": max_judge_calls,
            },
        )

    # ── Context manager ────────────────────────────────────────────────

    def __enter__(self) -> "LLMObservatory":
        return self

    def __exit__(self, exc_type, exc, tb) -> None:
        self.close()

    def close(self) -> None:
        """Release all lens handles — judge delegates, attention hooks, traces."""
        for lens_name in (
            "output",
            "attention",
            "retrieval",
            "agent",
            "alignment",
            "governance",
        ):
            lens = getattr(self, lens_name, None)
            if lens is not None and hasattr(lens, "close"):
                try:
                    lens.close()
                except Exception as exc:
                    # Cleanup path — zero-tolerance Rule 3 carve-out.
                    logger.warning(
                        "observatory.close.lens_error",
                        extra={"lens": lens_name, "error": str(exc)},
                    )

    # ── Composite report ──────────────────────────────────────────────

    def report(self, *, format: str = "rich") -> Any:
        """Composite Prescription Pad — one entry per lens.

        Args:
            format: ``"rich"`` (default) returns a Rich ``Table`` that
                renders as a colour-coded six-row table when printed in
                a notebook or terminal. ``"dict"`` returns the
                programmatic ``{lens: {summary, severity}}`` dict for
                callers that want to consume the data directly. ``"text"``
                returns a plain-text rendering for environments without
                Rich support.

        Each lens's ``report()`` returns a plain-text summary. The facade
        wraps each summary into a ``{"summary": str, "severity":
        HEALTHY/WARNING/CRITICAL/UNKNOWN}`` dict so downstream code (e.g.
        a dashboard widget) can colour-code per lens without parsing
        prose.

        The default switched from ``dict`` to ``rich`` in the M6 Ollama
        migration: students running ``obs.report()`` interactively in a
        notebook get a readable table instead of a wall of nested-dict
        text. Programmatic callers should pass ``format="dict"``
        explicitly.
        """
        result: dict[str, Any] = {}
        for lens_name in (
            "output",
            "attention",
            "retrieval",
            "agent",
            "alignment",
            "governance",
        ):
            lens = getattr(self, lens_name)
            try:
                summary = lens.report()
            except Exception as exc:
                logger.exception(
                    "observatory.report.lens_error",
                    extra={"lens": lens_name, "error": str(exc)},
                )
                result[lens_name] = {
                    "summary": f"{lens_name}-lens: error — {exc}",
                    "severity": _SEVERITY_UNKNOWN,
                }
                continue
            result[lens_name] = {
                "summary": summary,
                "severity": _derive_severity(lens_name, summary, lens),
            }
        logger.info(
            "observatory.report",
            extra={
                "run_id": self._run_id,
                "lenses": list(result.keys()),
                "source": "local_metric",
                "mode": "real",
                "format": format,
            },
        )

        if format == "dict":
            return result
        if format == "text":
            return _format_report_text(result)
        if format == "rich":
            return _format_report_rich(result)
        raise ValueError(
            f"unknown report format {format!r} — use 'rich', 'dict', or 'text'"
        )

    # ── Composite dashboard ───────────────────────────────────────────

    def plot_dashboard(self) -> go.Figure:
        """2x3 subplot — one panel per lens.

        Each lens provides its own plot method. The facade stitches the
        traces together into a single figure so M6 exercises can call
        ``obs.plot_dashboard().show()`` and see every lens at once.

        Lenses without data render the shared ``empty_figure`` placeholder.
        """
        from plotly.subplots import make_subplots

        fig = make_subplots(
            rows=2,
            cols=3,
            subplot_titles=(
                "Output (Stethoscope)",
                "Attention (X-Ray)",
                "Retrieval (Endoscope)",
                "Agent (Black Box)",
                "Alignment (ECG)",
                "Governance (Flight Recorder)",
            ),
            specs=[
                [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
                [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
            ],
        )

        # Attempt per-lens plotting; any failure yields a muted annotation
        # rather than crashing the whole dashboard.
        panel_specs = [
            ("output", 1, 1, self._output_panel),
            ("attention", 1, 2, self._attention_panel),
            ("retrieval", 1, 3, self._retrieval_panel),
            ("agent", 2, 1, self._agent_panel),
            ("alignment", 2, 2, self._alignment_panel),
            ("governance", 2, 3, self._governance_panel),
        ]
        for lens_name, row, col, builder in panel_specs:
            try:
                traces = builder()
            except Exception as exc:
                logger.warning(
                    "observatory.plot.panel_error",
                    extra={"lens": lens_name, "error": str(exc)},
                )
                traces = []
            for tr in traces:
                fig.add_trace(tr, row=row, col=col)

        fig.update_layout(
            title="LLM Observatory — six-lens dashboard",
            template=TEMPLATE,
            showlegend=False,
            height=800,
        )
        return fig

    # ── Per-lens panel builders (lightweight — just return traces) ────

    def _output_panel(self) -> list[Any]:
        import plotly.graph_objects as go  # local import — lazy

        df = self.output.results_df()
        if not df.height:
            return []
        return [
            go.Histogram(
                x=df["score"].to_list(),
                marker_color=PRIMARY,
                nbinsx=20,
                showlegend=False,
            )
        ]

    def _attention_panel(self) -> list[Any]:
        import plotly.graph_objects as go

        # Attention lens stores logits sweeps + probe readings; just count.
        n_attn = len(getattr(self.attention, "_attention_log", []))
        n_logit = len(getattr(self.attention, "_logit_log", []))
        n_probe = len(getattr(self.attention, "_probe_log", []))
        n_sae = len(getattr(self.attention, "_sae_log", []))
        return [
            go.Bar(
                x=["attention", "logit_lens", "probe", "sae"],
                y=[n_attn, n_logit, n_probe, n_sae],
                marker_color=ACCENT,
                showlegend=False,
            )
        ]

    def _retrieval_panel(self) -> list[Any]:
        import plotly.graph_objects as go

        df = self.retrieval.eval_df() if hasattr(self.retrieval, "eval_df") else None
        if df is None or not df.height:
            return []
        return [
            go.Scatter(
                x=list(range(df.height)),
                y=df["recall_at_k"].to_list(),
                mode="lines+markers",
                marker=dict(color=PRIMARY),
                showlegend=False,
            )
        ]

    def _agent_panel(self) -> list[Any]:
        import plotly.graph_objects as go

        traces = getattr(self.agent, "_traces", {})
        if not traces:
            return []
        rows = [(rid, trace.total_cost_usd()) for rid, trace in traces.items()]
        return [
            go.Bar(
                x=[r[0] for r in rows],
                y=[r[1] for r in rows],
                marker_color=PRIMARY,
                showlegend=False,
            )
        ]

    def _alignment_panel(self) -> list[Any]:
        import plotly.graph_objects as go

        df = (
            self.alignment.training_df()
            if hasattr(self.alignment, "training_df")
            else None
        )
        if df is None or not df.height:
            return []
        return [
            go.Scatter(
                x=df["step"].to_list(),
                y=df["reward"].to_list(),
                mode="lines+markers",
                line=dict(color=PRIMARY),
                showlegend=False,
            )
        ]

    def _governance_panel(self) -> list[Any]:
        import plotly.graph_objects as go

        if self._governance_engine is None:
            return []
        try:
            df = self.governance.budget_consumption()
        except Exception:
            return []
        if not df.height:
            return []
        return [
            go.Bar(
                x=df["dimension"].to_list(),
                y=df["consumed"].to_list(),
                marker_color=ACCENT,
                showlegend=False,
            )
        ]


# ════════════════════════════════════════════════════════════════════════
# Severity derivation — deterministic thresholds over metric summaries
# ════════════════════════════════════════════════════════════════════════


_SEVERITY_STYLE = {
    _SEVERITY_HEALTHY: "green",
    _SEVERITY_WARNING: "yellow",
    _SEVERITY_CRITICAL: "red bold",
    _SEVERITY_UNKNOWN: "dim",
}

_LENS_TITLE = {
    "output": "Output (Stethoscope)",
    "attention": "Attention (X-Ray)",
    "retrieval": "Retrieval (Endoscope)",
    "agent": "Agent (Black Box)",
    "alignment": "Alignment (ECG)",
    "governance": "Governance (Flight Recorder)",
}


def _format_report_text(result: dict[str, Any]) -> str:
    """Render the per-lens report dict as plain text (no Rich dependency)."""
    lines = ["LLM Observatory Prescription Pad", "=" * 40]
    for lens_name, payload in result.items():
        title = _LENS_TITLE.get(lens_name, lens_name)
        sev = payload.get("severity", _SEVERITY_UNKNOWN)
        summary = payload.get("summary", "")
        lines.append(f"  [{sev:<8}] {title}")
        for line in summary.splitlines() or [""]:
            lines.append(f"             {line}")
    return "\n".join(lines)


def _format_report_rich(result: dict[str, Any]) -> Any:
    """Render the per-lens report dict as a Rich ``Table``.

    Falls back to plain text if Rich is not importable so notebook
    environments without the optional dep still get a readable result.
    """
    try:
        from rich.table import Table
        from rich.text import Text
    except ImportError:
        return _format_report_text(result)

    table = Table(
        title="LLM Observatory — Prescription Pad",
        show_header=True,
        header_style="bold cyan",
        title_style="bold",
        expand=False,
    )
    table.add_column("Lens", style="bold", no_wrap=True)
    table.add_column("Severity", justify="center", no_wrap=True)
    table.add_column("Summary", overflow="fold")
    for lens_name, payload in result.items():
        title = _LENS_TITLE.get(lens_name, lens_name)
        sev = payload.get("severity", _SEVERITY_UNKNOWN)
        summary = payload.get("summary", "")
        sev_text = Text(sev, style=_SEVERITY_STYLE.get(sev, ""))
        table.add_row(title, sev_text, summary)
    return table


def _derive_severity(lens_name: str, summary: str, lens: Any) -> str:
    """Map a lens's report summary into one of four severity levels.

    This is deterministic plumbing (explicit exception 4 — safety guards
    — in ``rules/agent-reasoning.md``): the LLM already produced the
    metrics; this helper turns the tabular reading into a single label.
    """
    # "No readings yet" is the universal empty-state marker across lenses.
    # Cover both phrasings: "no readings recorded yet." (output/interp/
    # retrieval/alignment) and "no runs captured yet." (agent).
    if not summary:
        return _SEVERITY_UNKNOWN
    lowered = summary.lower()
    if (
        "no readings recorded yet" in lowered
        or "no runs captured yet" in lowered
        or "no engine configured" in lowered
    ):
        return _SEVERITY_UNKNOWN

    # Lens-specific escalation signals.
    lower = summary.lower()
    if lens_name == "output":
        if "faithfulness below" in lower or "scored below 0.5" in lower:
            return _SEVERITY_WARNING
        return _SEVERITY_HEALTHY
    if lens_name == "retrieval":
        if "recall" in lower and "below" in lower:
            return _SEVERITY_WARNING
        return _SEVERITY_HEALTHY
    if lens_name == "agent":
        if "stuck loop detected" in lower:
            return _SEVERITY_CRITICAL
        if "error event" in lower:
            return _SEVERITY_WARNING
        return _SEVERITY_HEALTHY
    if lens_name == "alignment":
        if "drifted far" in lower or "hacking scan" in lower:
            return _SEVERITY_CRITICAL
        if "too weak" in lower:
            return _SEVERITY_WARNING
        return _SEVERITY_HEALTHY
    if lens_name == "governance":
        if "chain broken" in lower:
            return _SEVERITY_CRITICAL
        if "approaching cap" in lower or "some drills passed" in lower:
            return _SEVERITY_WARNING
        if "no engine configured" in lower:
            return _SEVERITY_UNKNOWN
        return _SEVERITY_HEALTHY
    if lens_name == "attention":
        if "not applicable" in lower:
            return _SEVERITY_UNKNOWN
        return _SEVERITY_HEALTHY
    return _SEVERITY_HEALTHY


# ── shared/mlfp06/diagnostics/__init__.py ──
"""LLM Observatory — six clinical lenses for LLM / agent / RAG / governance systems.

Mirrors the M5 Doctor's Bag (``kailash_ml.diagnostics.DLDiagnostics``) for
the M6 problem domain. A single :class:`LLMObservatory` facade composes six
lens classes that each answer exactly one diagnostic question:

    1. Output Lens       — is the generation coherent, faithful, on-task?
    2. Attention Lens    — what does the model attend to; what circuit answers?
    3. Retrieval Lens    — did we fetch the right context, did we use it?
    4. Agent Trace Lens  — what did the agent do, and where did it spend budget?
    5. Alignment Lens    — is the fine-tuning signal rewarding the right thing?
    6. Governance Lens   — is the system inside its envelope; is the audit intact?

Quick start::


    obs = LLMObservatory(delegate=my_delegate)
    obs.output.faithfulness(response, context)
    obs.retrieval.recall_at_k(queries, relevant_ids, k=5)
    obs.agent.tool_usage_summary(trace)
    obs.governance.verify_chain(supervisor.audit.to_list())
    print(obs.report())

All DataFrames are Polars. All plots are Plotly. LLM calls route through
Kaizen :class:`~kaizen_agents.Delegate` (never raw ``openai.*``).
"""


__all__ = [
    "LLMObservatory",
    "LLMDiagnostics",
    "InterpretabilityDiagnostics",
    "RAGDiagnostics",
    "AgentDiagnostics",
    "AgentTrace",
    "AlignmentDiagnostics",
    "GovernanceDiagnostics",
    "JudgeVerdict",
]


# ── shared/mlfp06/ex_2.py ──
"""
Shared infrastructure for Exercise 2 — LLM Fine-Tuning.

Contains: SFT dataset loader (IMDB -> instruction pairs), LoRA/adapter
parameter-counting helpers, kailash-align config factory, base
hyperparameters, and output directory. Technique-specific classes
(LoRALayer, AdapterLayer, etc.) live in the per-technique files.
"""

import os
from pathlib import Path

import polars as pl
import torch


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
device = get_device()

# Output directory for all visualisation and training artifacts
REPO_ROOT = Path.cwd()
OUTPUT_DIR = REPO_ROOT / "outputs" / "ex2_finetuning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# SHARED HYPERPARAMETERS
# ════════════════════════════════════════════════════════════════════════

D_MODEL = 512  # synthetic demo hidden dim for from-scratch LoRA / adapter
LORA_RANK = 8  # default LoRA rank for the demos
LORA_ALPHA = 16.0  # typical alpha = 2 * rank
ADAPTER_BOTTLENECK = 64  # default adapter bottleneck

# ════════════════════════════════════════════════════════════════════════
# SFT DATA LOADING — IMDB instruction-response pairs
# ════════════════════════════════════════════════════════════════════════

DATA_DIR = REPO_ROOT / "data" / "mlfp06" / "imdb"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = DATA_DIR / "imdb_sft_2k.parquet"


def load_imdb_sft(
    n_rows: int = 2000, train_frac: float = 0.9
) -> tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    """Load IMDB movie reviews and reformat as SFT instruction pairs.

    Returns:
        (full, train, eval) polars DataFrames with columns:
            instruction, response, text, label
    """
    if CACHE_FILE.exists():
        print(f"Loading cached IMDB SFT pairs from {CACHE_FILE}")
        sft_data = pl.read_parquet(CACHE_FILE)
    else:
        print("Downloading stanfordnlp/imdb from HuggingFace (first run)...")
        from datasets import load_dataset

        ds = load_dataset("stanfordnlp/imdb", split="train")
        ds = ds.shuffle(seed=42).select(range(min(n_rows, len(ds))))

        label_names = {0: "negative", 1: "positive"}
        rows = []
        for row in ds:
            review = row["text"][:1500]
            sentiment = label_names[row["label"]]
            rows.append(
                {
                    "instruction": (
                        "Classify the sentiment of the following movie review as "
                        "either 'positive' or 'negative', then briefly justify "
                        f"your answer.\n\nReview: {review}"
                    ),
                    "response": (
                        f"Sentiment: {sentiment}. The reviewer expresses a clearly "
                        f"{sentiment} reaction to the film."
                    ),
                    "text": review,
                    "label": sentiment,
                }
            )
        sft_data = pl.DataFrame(rows)
        sft_data.write_parquet(CACHE_FILE)
        print(f"Cached {sft_data.height} SFT pairs to {CACHE_FILE}")

    n_train = int(sft_data.height * train_frac)
    train_data = sft_data[:n_train]
    eval_data = sft_data[n_train:]
    print(
        f"IMDB SFT loaded: {sft_data.height} pairs "
        f"({train_data.height} train / {eval_data.height} eval)"
    )
    return sft_data, train_data, eval_data


# ════════════════════════════════════════════════════════════════════════
# PARAMETER COUNTING HELPERS
# ════════════════════════════════════════════════════════════════════════


def count_lora_params(
    d_model: int, rank: int, num_modules: int = 1, num_layers: int = 1
) -> int:
    """LoRA trainable params = 2 * d * r per target module per layer.

    For d=512, r=8, modules=1, layers=1 -> 8,192 params.
    """
    return (d_model * rank + rank * d_model) * num_modules * num_layers


def count_adapter_params(d_model: int, bottleneck: int, num_layers: int = 1) -> int:
    """Adapter params per layer: down(d*b+b) + up(b*d+d) + layernorm(2*d).

    Matches the AdapterLayer in 02_adapter_from_scratch.py.
    """
    return (
        d_model * bottleneck + bottleneck + bottleneck * d_model + d_model + 2 * d_model
    ) * num_layers


def full_finetune_params(
    d_model: int, num_modules: int = 1, num_layers: int = 1
) -> int:
    """Full fine-tuning baseline: d^2 per module per layer."""
    return d_model * d_model * num_modules * num_layers


# ════════════════════════════════════════════════════════════════════════
# KAILASH-ALIGN CONFIG FACTORY
# ════════════════════════════════════════════════════════════════════════


def get_base_model_name() -> str:
    """Return the HuggingFace repo id of the SFT base model.

    Resolution: ``SFT_BASE_MODEL`` env var → ``Qwen/Qwen2.5-0.5B-Instruct``.
    Note: Align/TRL training needs a HuggingFace repo id, NOT an Ollama tag —
    the Ollama-side ``OLLAMA_FT_BASE_MODEL`` is for inference of the
    fine-tuned model after GGUF export, not for training.
    """
    return os.environ.get("SFT_BASE_MODEL") or "Qwen/Qwen2.5-0.5B-Instruct"


def build_sft_config(
    base_model: str | None = None,
    lora_r: int = 16,
    lora_alpha: int = 32,
    num_epochs: int = 3,
    output_subdir: str = "sft_output",
):
    """Build a kailash-align AlignmentConfig for SFT+LoRA training.

    Imported lazily so technique files that do not call SFT are not
    forced to install the align extra at import time.

    kailash-align 0.6.0+ uses a composed AlignmentConfig: top-level
    method + base_model_id + experiment_dir, plus required LoRAConfig
    + SFTConfig + DPOConfig sub-configs. The DPOConfig is unused for
    SFT but the constructor still expects it (default values are fine).
    """
    from kailash_align import AlignmentConfig, DPOConfig, LoRAConfig, SFTConfig

    return AlignmentConfig(
        method="sft",
        base_model_id=base_model or get_base_model_name(),
        lora=LoRAConfig(
            rank=lora_r,
            alpha=lora_alpha,
            target_modules=("q_proj", "v_proj"),
            dropout=0.05,
        ),
        sft=SFTConfig(
            num_train_epochs=num_epochs,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=4,
            learning_rate=2e-4,
            warmup_ratio=0.1,
            max_seq_length=512,
        ),
        dpo=DPOConfig(),
        experiment_dir=str(OUTPUT_DIR / output_subdir),
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP06 — Exercise 2.6: SFT with kailash-align AlignmentPipeline
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Build an AlignmentConfig for SFT + LoRA (no raw transformers.Trainer)
#   - Run AlignmentPipeline.train() on the IMDB instruction dataset
#   - Register the trained adapter in AdapterRegistry
#   - Visualise training loss + interpret train/eval gap
#   - Apply adapter-registry discipline to a Singapore e-commerce scenario
#
# PREREQUISITES: Exercises 2.1-2.5
# ESTIMATED TIME: ~40 min (training dominates)
#
# FRAMEWORK-FIRST: kailash-align, NOT raw transformers.Trainer.
#
# TASKS:
#   1. THEORY: why AlignmentPipeline beats raw SFTTrainer
#   2. BUILD: AlignmentConfig for SFT + LoRA r=16
#   3. TRAIN: pipeline.train() on IMDB instruction pairs
#   4. VISUALISE: train vs eval loss curve
#   5. APPLY: Singapore e-commerce adapter registry governance
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations


import matplotlib.pyplot as plt
from dotenv import load_dotenv


load_dotenv()



## THEORY — Why AlignmentPipeline (Framework-First)


In [ ]:
# Raw transformers.Trainer + peft.get_peft_model loses three guarantees:
#   1. LoRA config validation against the base model's module tree
#   2. Structured adapter artefact lifecycle (not a bag of .bin files)
#   3. AdapterRegistry hand-off between training and serving
# Framework-first rule: Engine layer first, primitives only when the
# engine cannot express the behaviour.



## TASK 1 — Load the SFT dataset


In [ ]:
print("\n" + "=" * 70)
print("TASK 1: Load IMDB SFT instruction pairs")
print("=" * 70)

# TODO: sft_data, train_data, eval_data = load_imdb_sft()
sft_data, train_data, eval_data = ____
print(f"Train: {train_data.height} pairs")
print(f"Eval:  {eval_data.height} pairs")



### Checkpoint 1


In [ ]:
assert train_data.height > 0, "Task 1: train split should not be empty"
assert eval_data.height > 0, "Task 1: eval split should not be empty"
print("✓ Checkpoint 1 passed — SFT data loaded\n")



## TASK 2 — BUILD: AlignmentConfig via the shared factory


In [ ]:
print("=" * 70)
print("TASK 2: Build AlignmentConfig for SFT + LoRA r=16")
print("=" * 70)

# TODO: Read the base model name from the environment via get_base_model_name()
base_model = ____
# TODO: Build an AlignmentConfig via build_sft_config(base_model=base_model,
# lora_r=16, lora_alpha=32, num_epochs=3, output_subdir="sft_output")
config = ____

print(f"  Method:        {config.method}")
print(f"  Base model:    {config.base_model_id}")
print(f"  LoRA:          r={config.lora.rank}, alpha={config.lora.alpha}")
print(f"  Target modules: {config.lora.target_modules}")
print(f"  Epochs:        {config.sft.num_train_epochs}")
print(f"  Batch size:    {config.sft.per_device_train_batch_size}")
print(f"  Learning rate: {config.sft.learning_rate}")
print(f"  Output dir:    {config.experiment_dir}")



### Checkpoint 2


In [ ]:
assert config.method == "sft", "Task 2: method should be 'sft'"
assert config.lora.rank == 16, "Task 2: LoRA rank should be 16"
assert "q_proj" in config.lora.target_modules, "Task 2: q_proj must be a target module"
print("✓ Checkpoint 2 passed — AlignmentConfig built\n")



## TASK 3 — TRAIN: AlignmentPipeline + register adapter


Train the SFT adapter and register it. Returns metrics dict.


In [ ]:
# Set MLFP_SKIP_SFT_TRAIN=1 to exercise the registry with synthetic
# metrics on environments without GPU / HF access.

print("=" * 70)
print("TASK 3: AlignmentPipeline.train() -> AdapterRegistry")
print("=" * 70)


async def run_sft_and_register() -> dict:
    import os

    skip = os.environ.get("MLFP_SKIP_SFT_TRAIN") == "1"

    if skip:
        print("  MLFP_SKIP_SFT_TRAIN=1 -> using synthetic metrics")
        metrics = {
            "final_loss": 0.742,
            "eval_loss": 0.881,
            "training_time_seconds": 0.0,
            "adapter_path": str(OUTPUT_DIR / "sft_output" / "adapter"),
        }
    else:
        from kailash_align import (
            AdapterRegistry,
            AdapterSignature,
            AlignmentPipeline,
        )

        # TODO: Instantiate AlignmentPipeline(config) and await
        # pipeline.train(train_data=..., eval_data=...)
        pipeline = ____
        print("  Running SFT training (this may take several minutes)...")
        result = ____

        metrics = {
            "final_loss": result.final_loss,
            "eval_loss": result.eval_loss,
            "training_time_seconds": result.training_time_seconds,
            "adapter_path": result.adapter_path,
        }
        print(f"  Final loss:    {metrics['final_loss']:.4f}")
        print(f"  Eval loss:     {metrics['eval_loss']:.4f}")
        print(f"  Training time: {metrics['training_time_seconds']:.0f}s")

        # TODO: Instantiate registry = AdapterRegistry()
        registry = ____
        # TODO: Build an AdapterSignature with base_model_id=config.base_model_id,
        #   adapter_type="lora", training_method="sft"
        signature = ____
        # TODO: await registry.register_adapter(
        #     name="imdb_sentiment_sft_v1",
        #     adapter_path=metrics["adapter_path"],
        #     signature=signature,
        #     training_metrics={"final_loss": ..., "eval_loss": ...},
        #     tags=["imdb", "sentiment", "lora-r16"])
        # register_adapter returns an AdapterVersion (not a string).
        version = ____
        # TODO: Build a stable adapter_id string of the form
        #   f"{version.adapter_name}:v{version.version}"
        adapter_id = ____
        metrics["adapter_id"] = adapter_id
        print(f"  Registered as: {adapter_id}")

    return metrics


metrics  = await run_sft_and_register()



### Checkpoint 3


In [ ]:
assert metrics["final_loss"] is not None, "Task 3: training should produce a loss"
assert metrics["final_loss"] > 0, "Task 3: final loss should be positive"
print("✓ Checkpoint 3 passed — SFT training + registration complete\n")



## TASK 4 — VISUALISE: train vs eval loss bars


In [ ]:
print("=" * 70)
print("TASK 4: Visualise final train vs eval loss")
print("=" * 70)

labels = ["Train loss (final)", "Eval loss"]
values = [metrics["final_loss"], metrics["eval_loss"]]

# TODO: Bar chart with train and eval loss, annotations on each bar.
# Save to OUTPUT_DIR / "ex2_sft_train_eval.png"
____
fname = OUTPUT_DIR / "ex2_sft_train_eval.png"
print(f"  Saved: {fname}")

gap = metrics["eval_loss"] - metrics["final_loss"]
print(f"\n  Train-eval gap: {gap:+.3f}")
print("  Healthy: |gap| < 0.15; larger values suggest over/under-fitting")



### Checkpoint 4


In [ ]:
assert fname.exists(), "Task 4: loss plot should exist"
print("✓ Checkpoint 4 passed — train vs eval visualised\n")



## TASK 5 — APPLY: Singapore e-commerce adapter registry governance


In [ ]:
# A Singapore e-commerce platform has 37 LoRAs trained over 18 months
# with no shared record of "which adapter is live right now". Customer
# complaints cannot be traced back to a training run, compliance
# audits fail, rollback takes hours. Fix: mandate AdapterRegistry
# entries for every training run with name, base SHA, method, metrics,
# and tags. Serving pulls by registry name, not filename.

print("Singapore e-commerce adapter governance:")
# TODO: Compute annual_retraining_saving (16 runs * S$600), engineer hours
# (16 * 8), annual_engineer_saving at S$90/hr, and total_annual
annual_retraining_saving = ____
annual_engineer_hours_saved = ____
engineer_hourly_sgd = 90
annual_engineer_saving = ____
total_annual = ____
print(f"  Retraining runs avoided / year:      {4 * 4}")
print(f"  Annual retraining cost avoided:      S${annual_retraining_saving:,}")
print(f"  Engineer hours saved / year:         {annual_engineer_hours_saved}")
print(f"  Annual engineer time saving:         S${annual_engineer_saving:,}")
print(f"  Total direct annual saving:          S${total_annual:,}")
print(f"  Plus: MAS compliance + rollback SLA (unquantified but critical)")
print(f"  Recommended: AdapterRegistry mandatory for all SFT runs")



### Checkpoint 5


In [ ]:
assert total_annual > 0, "Task 5: governance should deliver positive ROI"
print("✓ Checkpoint 5 passed — e-commerce governance analysed\n")



## REFLECTION


[x] Built an AlignmentConfig for SFT + LoRA via kailash-align
      (framework-first: no raw transformers.Trainer)
  [x] Ran AlignmentPipeline.train() end-to-end on IMDB SFT data
  [x] Registered the trained adapter in AdapterRegistry with metrics
  [x] Visualised train vs eval loss and interpreted the gap
  [x] Applied adapter-registry governance to a Singapore e-commerce
      scenario (~S$32k/year saving + unblocked MAS compliance)

  KEY INSIGHT: SFT is the first rung of the alignment ladder.
  AlignmentPipeline + AdapterRegistry give you the versioning and
  audit trail you need before you layer DPO / GRPO / RLHF on top.

  Next: Exercise 3 (DPO Alignment) moves from "learn the right
  response" to "learn which response is PREFERRED".


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — six lenses before completion


In [ ]:
# The LLM Observatory extends M5's Doctor's Bag for LLM/agent work.
# Six lenses:
#   1. Output        — is the generation coherent, factual, on-task?
#   2. Attention     — what does the model attend to internally?
#   3. Retrieval     — did we fetch the right context?  [RAG only]
#   4. Agent Trace   — what did the agent actually do?  [Agent only]
#   5. Alignment     — is it aligned with our intent?   [Fine-tune only]
#   6. Governance    — is it within policy?            [PACT only]

# Primary lens: Alignment (KL divergence from base, reward margin).
# Secondary: Output (judge quality on paired completions), Attention
# (layer-wise shift in target modules for LoRA).
if False:  # scaffold — requires trained base + adapter checkpoint
    obs = LLMObservatory(run_id="ex_2_finetune_run")
    # Typical alignment read:
    # for step, metrics in enumerate(training_log):
    #     obs.alignment.log_training_step(step=step, **metrics)
    # obs.alignment.evaluate_pair(base_responses, adapter_responses)
    print("\n── LLM Observatory Report ──")
    findings = obs.report()

# ══════ EXPECTED OUTPUT (synthesised reference) ══════



## LLM Observatory — composite Prescription Pad


In [ ]:
#   [!] Alignment  (WARNING): KL divergence from base = 0.42 nats
#       Fix: healthy range 0.2-1.0; this is low-end — adapter barely
#            moved. Increase LoRA rank or learning rate.
#   [✓] Output     (HEALTHY): judge win-rate 0.58 vs base (>0.50 = good)
#   [✓] Attention  (HEALTHY): shift concentrated in q_proj/v_proj as
#       expected for LoRA; no drift in frozen layers.
#   [?] Retrieval / Agent / Governance (n/a)



## STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:

 [ALIGNMENT LENS] KL 0.42 nats is the SIGNATURE of a cautiously-trained
    LoRA adapter — it diverged from the base distribution but not
    enough to break it. Above 2.0 nats signals over-fit; below 0.2
    signals the adapter barely learned. Our value is slightly under the
    0.5 floor we want for visible task lift.
    >> Prescription: raise lora_r from 8 -> 16 or train another epoch.
 [OUTPUT LENS] Win-rate 0.58 > 0.50 confirms the adapter is better
    than base on held-out prompts — tiny lift but statistically real.
 [ATTENTION LENS] Shift localised in the target modules = LoRA is
    doing what it's supposed to do (low-rank delta on attention
    projections, frozen MLP). If attention shifted everywhere you'd
    know you accidentally unfroze a module.
